# 任务4：最小补跑、固定OOF融合诊断与明确备份

**仅在要继续做融合诊断、且旧权重/概率无法找回时运行。已有提交CSV和实验记录仍然有效，直接使用已有CSV不需要重新训练。** 本文件不要求旧模型或分析包，只需原比赛数据、GPU以及官方ImageNet初始权重（可在线下载）。

只重建两个模型系列：原尺寸干净标签模型、512 L1_proxy，各5折80轮。沿用原实现、seed42、划分、训练预算和最终EMA。**不重跑L0、L2、EMA/膨胀消融或clDice。** 原始三个ML模块保持不变。新运行可能因运行环境/随机计算产生差异，不能保证重建出原权重哈希或原提交；只用本轮完整OOF评价，不混用旧分数挑模型。

按原T4日志，两组训练/监控合计约一小时，最终OOF推理、检查点保存和备份另计。开始运行即会重新训练，不是恢复已消失的文件。

## 先固定诊断规则

两个单模型：A为原尺寸D4，阈值0.1；B为L1 D4，阈值0.4。这两个阈值来自已有OOF实验，存在开发选择偏差；不是独立验证，也不确认隐藏真值。

唯一新增规则：先对两个概率取logit，分别减去logit(0.1)/logit(0.4)，再等权平均并sigmoid，最后固定0.5二值化。概率裁剪epsilon=1e-6。**这是阈值中心对齐的固定启发式，不是已校准概率，不保证融合更好。** 本轮不搜索融合权重、阈值或种子。

同一预测同时评价C/P，给出逐图/逐折、最差折、像素和骨架指标，以及两模型错误交集、A错B对/B错A对的像素数。每张图只用对应未见过它的折模型；不能把另外四折模型也用于该图OOF。只跑这一个融合规则，无测试预测、比赛CSV或自动赢家。

## 备份：必须完成下载

每完成一折自动更新三份ZIP并显示链接；这仍是临时目录中的文件，**生成ZIP不等于已经永久保存**。

| 文件 | 用途 | 能否恢复训练 |
|---|---|---|
| task4_recovery_full.zip | 模型、优化器、EMA、概率和官方初始权重 | 可以 |
| task4_recovery_oof.zip | OOF概率、配置和记录，后续诊断所需 | 不可以 |
| task4_recovery_analysis.zip | 轻量指标和源码，发回分析 | 不可以 |

结束前下载前两份到本机，并确认浏览器下载完成、文件大小非0；第三份发回分析。完整输出约5GB，ZIP打包需要额外磁盘空间；空间不足时明确提示完整备份未生成，不把OOF包冒充完整备份。

也可以在训练完成后Quick Save，并在Advanced Settings显式选择Always save output when creating a Quick Save，然后到该版本Output确认文件存在；参考[Kaggle官方文档](https://www.kaggle.com/docs/notebooks)及[Kaggle工作人员说明](https://www.kaggle.com/product-feedback/296805)。这次建议同时保留本机备份。无需为了保存已生成文件再次Save & Run All重训。


## 1. 路径与开关


In [ ]:
from pathlib import Path
DATA_DIR = None   # 原任务4比赛数据；不需要旧模型或旧概率
OUTPUT_DIR = Path('/kaggle/working/task4_recovery') if Path('/kaggle/working').exists() else Path.cwd()/'task4_recovery_output'
RESUME_FROM = None  # 仅恢复本轮的完整输出/task4_recovery_full.zip；不是轻量分析或OOF ZIP
WEIGHTS_PATH = None # 无网络时填写官方resnet34-b627a593.pth
FOLDS_TO_RUN = [0,1,2,3,4]
RUN_TRAINING = True
SAVE_FULL_BACKUP = True
INSTALL_MISSING_DEPS = True


## 2. 加载内嵌实现


In [ ]:
import importlib.util,sys,subprocess
if importlib.util.find_spec('skimage') is None:
    if not INSTALL_MISSING_DEPS:raise RuntimeError('Install scikit-image or enable INSTALL_MISSING_DEPS.')
    subprocess.check_call([sys.executable,'-m','pip','install','-q','scikit-image'])
RUNTIME_DIR=(Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd())/'task4_recovery_runtime'
RUNTIME_DIR.mkdir(parents=True,exist_ok=True)
SOURCES={'task4_pipeline.py': '"""Task 4: small-data retinal vessel segmentation, audit through submission.\n\nOnly the supplied training images/labels are used. No downloaded weights.\nThe original segmentation_to_csv.py is kept unchanged and checked on export.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport hashlib\nimport importlib.util\nimport json\nimport math\nimport os\nimport random\nimport time\nfrom dataclasses import asdict, dataclass\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\nimport torch\nfrom PIL import Image, ImageDraw\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom torch.utils.data import DataLoader, Dataset\n\n\n@dataclass\nclass Config:\n    seed: int = 42\n    epochs: int = 80\n    size: int = 512\n    patch: int = 256\n    patches_per_epoch: int = 128\n    batch_size: int = 8\n    base_channels: int = 24\n    lr: float = 0.001\n    weight_decay: float = 0.0001\n    folds: int = 5\n    fold: int = 0\n    threads: int = 4\n    amp: bool = True\n\n\ndef save_json(path, obj):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_suffix(path.suffix + \'.tmp\')\n    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding=\'utf-8\')\n    os.replace(tmp, path)\n\n\ndef file_sha(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef seed_all(seed):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n\n\ndef find_data(data_dir=None):\n    if data_dir:\n        candidates = [Path(data_dir)]\n    else:\n        base = Path(__file__).resolve().parent if \'__file__\' in globals() else Path.cwd()\n        candidates = [base / \'2026task-4/segmentation\', Path.cwd() / \'2026task-4/segmentation\',\n                      Path.cwd() / \'task4/2026task-4/segmentation\']\n        if Path(\'/kaggle/input\').exists():\n            candidates += [p.parent for p in Path(\'/kaggle/input\').rglob(\'sample_submission.csv\')]\n    found = {p.resolve() for p in candidates if (p / \'train/label\').is_dir()\n             and (p / \'test/image\').is_dir() and (p / \'sample_submission.csv\').is_file()}\n    if len(found) != 1:\n        raise ValueError(f\'Set data_dir to the segmentation folder. Found: {sorted(map(str, found))}\')\n    return found.pop()\n\n\ndef list_images(directory):\n    files = sorted(Path(directory).glob(\'*.jpg\'), key=lambda p: int(p.stem))\n    if not files:\n        raise ValueError(f\'No jpg images: {directory}\')\n    return files\n\n\ndef strict_label(path, size=512):\n    # JPEG labels contain ringing; >0 would thicken the foreground substantially.\n    mask = (np.asarray(Image.open(path).convert(\'L\')) > 127).astype(np.uint8)\n    return cv2.resize(mask, (size, size), interpolation=cv2.INTER_NEAREST).astype(bool)\n\n\ndef converter_label(path):\n    # Exact target IF hidden ground truth was processed by the supplied converter.\n    # This is a diagnostic proxy, not an assertion about the private scoring code.\n    return np.asarray(Image.open(path).convert(\'L\').resize((512, 512), Image.Resampling.BILINEAR)) > 0\n\n\ndef make_features(path, size=512):\n    rgb = np.asarray(Image.open(path).convert(\'RGB\'))\n    rgb = cv2.resize(rgb, (size, size), interpolation=cv2.INTER_AREA)\n    green = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(rgb[:, :, 1])\n    return np.concatenate([rgb, green[:, :, None]], axis=2)\n\n\ndef audit(root, output, cfg):\n    output.mkdir(parents=True, exist_ok=True)\n    train, test = list_images(root / \'train/image\'), list_images(root / \'test/image\')\n    labels = {p.stem: p for p in list_images(root / \'train/label\')}\n    if len(train) != 20 or len(test) != 20 or set(labels) != {p.stem for p in train}:\n        raise ValueError(\'Expected 20 train image/label pairs and 20 test images.\')\n    with (root / \'sample_submission.csv\').open(newline=\'\', encoding=\'utf-8-sig\') as f:\n        reader = csv.DictReader(f)\n        if reader.fieldnames != [\'Id\', \'Predicted\']:\n            raise ValueError(f\'Unexpected template columns: {reader.fieldnames}\')\n        ids = [r[\'Id\'] for r in reader]\n    if len(ids) != len(set(ids)) or set(ids) != {p.stem for p in test}:\n        raise ValueError(\'Template IDs do not match test images.\')\n    rows, groups, test_hashes = [], {}, set()\n    for split, paths in [(\'train\', train), (\'test\', test)]:\n        for p in paths:\n            im = Image.open(p).convert(\'RGB\')\n            digest = hashlib.sha256(np.asarray(im).tobytes()).hexdigest()\n            row = dict(split=split, id=p.stem, width=im.width, height=im.height,\n                       image_sha256=file_sha(p), pixel_sha256=digest)\n            if split == \'train\':\n                if Image.open(labels[p.stem]).size != im.size:\n                    raise ValueError(f\'Image/label size mismatch: {p.stem}\')\n                row.update(label_sha256=file_sha(labels[p.stem]),\n                           vessel_fraction=float(strict_label(labels[p.stem]).mean()),\n                           converter_fraction=float(converter_label(labels[p.stem]).mean()))\n                groups.setdefault(digest, []).append(p.stem)\n            else:\n                test_hashes.add(digest)\n            rows.append(row)\n    if set(groups) & test_hashes:\n        raise ValueError(\'Exact decoded train/test duplicate detected; inspect before training.\')\n    ordered = sorted(groups)\n    random.Random(cfg.seed).shuffle(ordered)\n    assignments = {}\n    sizes = [0] * cfg.folds\n    for digest in ordered:\n        fold = min(range(cfg.folds), key=lambda k: sizes[k])\n        for image_id in groups[digest]:\n            assignments[image_id] = fold\n        sizes[fold] += len(groups[digest])\n    for row in rows:\n        row[\'fold\'] = assignments.get(row[\'id\'], -1) if row[\'split\'] == \'train\' else -1\n    fields = sorted({key for row in rows for key in row})\n    with (output / \'manifest.csv\').open(\'w\', newline=\'\', encoding=\'utf-8\') as f:\n        writer = csv.DictWriter(f, fields)\n        writer.writeheader()\n        writer.writerows(rows)\n    manifest_fingerprint = hashlib.sha256(json.dumps(rows, sort_keys=True).encode()).hexdigest()\n    summary = dict(train_images=len(train), test_images=len(test), unique_train_groups=len(groups),\n                   folds=sizes, validation_ids=[p.stem for p in train if assignments[p.stem] == cfg.fold],\n                   decoded_train_test_overlap=0, patient_ids_available=False,\n                   data_fingerprint=manifest_fingerprint,\n                   observed_label_polarity=\'white vessels / black background\',\n                   screenshot_label_polarity=\'black vessels / white background\',\n                   converter=dict(size=[512, 512], resize=\'PIL bilinear\', foreground=\'nonzero\',\n                                  flatten=\'row-major C order\', start_index=1,\n                                  source_sha256=file_sha(root / \'segmentation_to_csv.py\')),\n                   strict_label=\'native grayscale >127, then nearest resize\',\n                   mean_clean_foreground=float(np.mean([r[\'vessel_fraction\'] for r in rows if r[\'split\']==\'train\'])),\n                   mean_converter_foreground=float(np.mean([r[\'converter_fraction\'] for r in rows if r[\'split\']==\'train\'])))\n    save_json(output / \'audit_summary.json\', summary)\n    return train, test, labels, assignments, ids, summary\n\n\ndef paired_d4(image, mask, rotation, flip):\n    image, mask = np.rot90(image, rotation), np.rot90(mask, rotation)\n    if flip:\n        image, mask = np.flip(image, 1), np.flip(mask, 1)\n    return image.copy(), mask.copy()\n\n\nclass PatchDataset(Dataset):\n    def __init__(self, paths, labels, cfg):\n        self.cfg = cfg\n        self.images = [make_features(p, cfg.size) for p in paths]\n        self.masks = [strict_label(labels[p.stem], cfg.size).astype(np.float32) for p in paths]\n        self.positive = [np.argwhere(m > 0) for m in self.masks]\n\n    def __len__(self):\n        return self.cfg.patches_per_epoch\n\n    def __getitem__(self, index):\n        idx = random.randrange(len(self.images))\n        image, mask = self.images[idx], self.masks[idx]\n        size, patch = self.cfg.size, self.cfg.patch\n        crop = random.randint(patch, min(size, int(patch * 1.25)))\n        if random.random() < 0.3 and len(self.positive[idx]):\n            y, x = self.positive[idx][random.randrange(len(self.positive[idx]))]\n            top, left = int(np.clip(y - crop//2, 0, size-crop)), int(np.clip(x-crop//2, 0, size-crop))\n        else:\n            top, left = random.randint(0, size-crop), random.randint(0, size-crop)\n        image = cv2.resize(image[top:top+crop, left:left+crop], (patch, patch), interpolation=cv2.INTER_LINEAR)\n        mask = cv2.resize(mask[top:top+crop, left:left+crop], (patch, patch), interpolation=cv2.INTER_NEAREST)\n        image, mask = paired_d4(image, mask, random.randrange(4), random.random() < 0.5)\n        image = image.astype(np.float32) / 255.0\n        image = np.clip(image * random.uniform(0.9, 1.1) + random.uniform(-0.04, 0.04), 0, 1)\n        image = image ** random.uniform(0.85, 1.15)\n        return torch.from_numpy(image.transpose(2, 0, 1).copy()) * 2 - 1, torch.from_numpy(mask[None].copy())\n\n\nclass ConvBlock(nn.Sequential):\n    def __init__(self, cin, cout):\n        super().__init__(nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.GroupNorm(8, cout), nn.SiLU(),\n                         nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.GroupNorm(8, cout), nn.SiLU())\n\n\nclass VesselUNet(nn.Module):\n    def __init__(self, base=24):\n        super().__init__()\n        ch = [base * (2**i) for i in range(5)]\n        self.enc = nn.ModuleList([ConvBlock(4 if i == 0 else ch[i-1], ch[i]) for i in range(5)])\n        self.dec = nn.ModuleList([ConvBlock(ch[i+1]+ch[i], ch[i]) for i in range(3, -1, -1)])\n        self.head = nn.Conv2d(base, 1, 1)\n\n    def forward(self, x):\n        skips = []\n        for i, block in enumerate(self.enc):\n            x = block(x if i == 0 else F.max_pool2d(x, 2))\n            skips.append(x)\n        for block, skip in zip(self.dec, reversed(skips[:-1])):\n            x = F.interpolate(x, size=skip.shape[-2:], mode=\'bilinear\', align_corners=False)\n            x = block(torch.cat([x, skip], 1))\n        return self.head(x)\n\n\ndef segmentation_loss(logits, target):\n    bce = F.binary_cross_entropy_with_logits(logits.float(), target.float())\n    prob = logits.float().sigmoid()\n    dice = (2*(prob*target).sum((1,2,3))+1) / (prob.sum((1,2,3))+target.sum((1,2,3))+1)\n    return 0.5*bce + 0.5*(1-dice.mean())\n\n\ndef dice_score(pred, target):\n    pred, target = np.asarray(pred, bool), np.asarray(target, bool)\n    return float((2*np.logical_and(pred, target).sum()+1e-7) / (pred.sum()+target.sum()+1e-7))\n\n\n@torch.inference_mode()\ndef predict(model, path, cfg, device):\n    model.eval()\n    image = make_features(path, cfg.size)\n    step = cfg.patch // 2\n    starts = list(range(0, cfg.size-cfg.patch+1, step))\n    if starts[-1] != cfg.size-cfg.patch:\n        starts.append(cfg.size-cfg.patch)\n    coords = [(y,x) for y in starts for x in starts]\n    weight = np.maximum(np.outer(np.hanning(cfg.patch), np.hanning(cfg.patch)), 0.05).astype(np.float32)\n    sums, weights = np.zeros((cfg.size, cfg.size), np.float32), np.zeros((cfg.size, cfg.size), np.float32)\n    for start in range(0, len(coords), cfg.batch_size):\n        positions = coords[start:start+cfg.batch_size]\n        patches = np.stack([image[y:y+cfg.patch, x:x+cfg.patch].transpose(2,0,1) for y,x in positions])\n        tensor = torch.from_numpy(patches.copy()).to(device).float()/127.5-1\n        with torch.autocast(device_type=device.type, enabled=cfg.amp and device.type==\'cuda\'):\n            probs = model(tensor).float().sigmoid()[:,0].cpu().numpy()\n        for (y,x), prob in zip(positions, probs):\n            sums[y:y+cfg.patch,x:x+cfg.patch] += prob * weight\n            weights[y:y+cfg.patch,x:x+cfg.patch] += weight\n    return sums/weights\n\n\ndef atomic_checkpoint(path, value):\n    tmp = path.with_suffix(\'.tmp\')\n    torch.save(value, tmp)\n    os.replace(tmp, path)\n\n\ndef rng_state():\n    return dict(python=random.getstate(), numpy=np.random.get_state(), torch=torch.get_rng_state(),\n                cuda=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None)\n\n\ndef restore_rng(state):\n    random.setstate(state[\'python\'])\n    np.random.set_state(state[\'numpy\'])\n    torch.set_rng_state(state[\'torch\'].cpu())\n    if torch.cuda.is_available() and state[\'cuda\'] is not None:\n        torch.cuda.set_rng_state_all([s.cpu() for s in state[\'cuda\']])\n\n\ndef train_stage(paths, val_paths, labels, cfg, folder, device, epochs):\n    folder.mkdir(parents=True, exist_ok=True)\n    seed_all(cfg.seed)\n    dataset = PatchDataset(paths, labels, cfg)\n    loader = DataLoader(dataset, batch_size=cfg.batch_size, num_workers=0, shuffle=False,\n                        pin_memory=device.type==\'cuda\')\n    model = VesselUNet(cfg.base_channels).to(device)\n    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)\n    # Full retraining retains the original 80-epoch cosine horizon.\n    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.epochs, eta_min=cfg.lr*0.02)\n    scaler = torch.amp.GradScaler(\'cuda\', enabled=cfg.amp and device.type==\'cuda\')\n    best, history, first = -1.0, [], 1\n    last = folder / \'last.pt\'\n    if last.exists():\n        ckpt = torch.load(last, map_location=\'cpu\', weights_only=False)\n        if ckpt[\'train_ids\'] != [p.stem for p in paths] or ckpt[\'target_epochs\'] != epochs:\n            raise ValueError(\'Stage mismatch. Use a new output directory.\')\n        model.load_state_dict(ckpt[\'model\'])\n        optimizer.load_state_dict(ckpt[\'optimizer\'])\n        scheduler.load_state_dict(ckpt[\'scheduler\'])\n        scaler.load_state_dict(ckpt[\'scaler\'])\n        first, best, history = ckpt[\'epoch\']+1, ckpt[\'best\'], ckpt[\'history\']\n        restore_rng(ckpt[\'rng\'])\n        print(f\'{folder.name}: resume at epoch {first}/{epochs}\', flush=True)\n    targets = {p.stem: strict_label(labels[p.stem], cfg.size) for p in val_paths}\n    for epoch in range(first, epochs+1):\n        begin = time.perf_counter()\n        model.train()\n        losses = []\n        lr = optimizer.param_groups[0][\'lr\']\n        for image, mask in loader:\n            image, mask = image.to(device, non_blocking=True), mask.to(device, non_blocking=True)\n            optimizer.zero_grad(set_to_none=True)\n            with torch.autocast(device_type=device.type, enabled=cfg.amp and device.type==\'cuda\'):\n                loss = segmentation_loss(model(image), mask)\n            if not torch.isfinite(loss):\n                raise FloatingPointError(\'Nonfinite training loss\')\n            scaler.scale(loss).backward()\n            scaler.unscale_(optimizer)\n            nn.utils.clip_grad_norm_(model.parameters(), 5.0)\n            scaler.step(optimizer)\n            scaler.update()\n            losses.append(float(loss.detach()))\n        scheduler.step()\n        score = float(np.mean([dice_score(predict(model,p,cfg,device) >= 0.5, targets[p.stem]) for p in val_paths])) if val_paths else None\n        improved = score is not None and score > best\n        if improved:\n            best = score\n            atomic_checkpoint(folder/\'best.pt\', dict(model=model.state_dict(), epoch=epoch, dice_05=score))\n        row = dict(epoch=epoch, loss=float(np.mean(losses)), dice_05=score, lr=lr,\n                   seconds=round(time.perf_counter()-begin, 3))\n        history.append(row)\n        atomic_checkpoint(last, dict(model=model.state_dict(), optimizer=optimizer.state_dict(),\n                          scheduler=scheduler.state_dict(), scaler=scaler.state_dict(), epoch=epoch,\n                          best=best, history=history, rng=rng_state(), train_ids=[p.stem for p in paths], target_epochs=epochs))\n        with (folder/\'history.csv\').open(\'w\', newline=\'\', encoding=\'utf-8\') as f:\n            writer = csv.DictWriter(f, list(row))\n            writer.writeheader()\n            writer.writerows(history)\n        print(f\'{folder.name} {epoch:03d}/{epochs} loss={row["loss"]:.4f}\' +\n              (f\' val_dice@0.5={score:.5f} best={best:.5f}\' if score is not None else \'\') +\n              f\' seconds={row["seconds"]}\', flush=True)\n    selected = torch.load(folder/(\'best.pt\' if val_paths else \'last.pt\'), map_location=\'cpu\', weights_only=False)\n    model.load_state_dict(selected[\'model\'])\n    return model, int(selected[\'epoch\'])\n\n\ndef rle_encode(mask):\n    flat = np.asarray(mask, dtype=np.uint8).ravel(order=\'C\')\n    changes = np.flatnonzero(np.diff(np.r_[np.uint8(0), flat, np.uint8(0)])) + 1\n    changes[1::2] -= changes[::2]\n    return \' \'.join(map(str, changes))\n\n\ndef rle_decode(encoded, shape=(512,512)):\n    values = np.array([int(s) for s in encoded.split()], dtype=np.int64)\n    if len(values)%2:\n        raise ValueError(\'Odd RLE length\')\n    mask = np.zeros(int(np.prod(shape)), dtype=bool)\n    previous_end = 0\n    for start, length in values.reshape(-1,2):\n        if start < 1 or length < 1 or start <= previous_end or start+length-1 > mask.size:\n            raise ValueError(\'RLE bounds/order/overlap violation\')\n        mask[start-1:start-1+length] = True\n        previous_end = start+length-1\n    return mask.reshape(shape)\n\n\ndef load_converter(root):\n    spec = importlib.util.spec_from_file_location(\'task4_official_converter\', root/\'segmentation_to_csv.py\')\n    module = importlib.util.module_from_spec(spec)\n    spec.loader.exec_module(module)\n    return module\n\n\ndef preview(paths, probs, labels, threshold, output):\n    tile = 256\n    canvas = Image.new(\'RGB\', (tile*3, (tile+24)*min(4,len(paths))), \'white\')\n    draw = ImageDraw.Draw(canvas)\n    for row,p in enumerate(paths[:4]):\n        rgb = Image.open(p).convert(\'RGB\').resize((tile,tile))\n        target = strict_label(labels[p.stem],tile) if labels else None\n        pred = cv2.resize(probs[p.stem],(tile,tile),interpolation=cv2.INTER_LINEAR) >= threshold\n        y = row*(tile+24)\n        canvas.paste(rgb,(0,y+24))\n        draw.text((4,y+4), f\'Image {p.stem}\',fill=\'black\')\n        if target is not None:\n            canvas.paste(Image.fromarray((target*255).astype(np.uint8)).convert(\'RGB\'),(tile,y+24))\n            draw.text((tile+4,y+4),\'Label: white vessels\',fill=\'black\')\n        else:\n            overlay = np.asarray(rgb).copy()\n            overlay[pred] = (0.4*overlay[pred]+0.6*np.array([0,255,100])).astype(np.uint8)\n            canvas.paste(Image.fromarray(overlay),(tile,y+24))\n            draw.text((tile+4,y+4),\'Vessel overlay\',fill=\'black\')\n        canvas.paste(Image.fromarray((pred*255).astype(np.uint8)).convert(\'RGB\'),(tile*2,y+24))\n        draw.text((tile*2+4,y+4),f\'Prediction t={threshold:.3f}\',fill=\'black\')\n    canvas.save(output)\n\n\ndef validation_report(model, paths, labels, cfg, device, output, epoch):\n    probs = {p.stem:predict(model,p,cfg,device) for p in paths}\n    np.savez_compressed(output/\'validation_probabilities.npz\', **probs)\n    thresholds = [round(float(t),3) for t in np.arange(0.2,0.801,0.025)]\n    rows = []\n    for threshold in thresholds:\n        strict = [dice_score(probs[p.stem]>=threshold, strict_label(labels[p.stem],cfg.size)) for p in paths]\n        proxy = [dice_score(probs[p.stem]>=threshold, converter_label(labels[p.stem])) for p in paths]\n        rows.append(dict(threshold=threshold, mean_vessel_dice=float(np.mean(strict)),\n                         converter_target_proxy_dice=float(np.mean(proxy))))\n    best = max(rows,key=lambda r:(r[\'mean_vessel_dice\'],-abs(r[\'threshold\']-0.5)))\n    threshold = best[\'threshold\']\n    with (output/\'threshold_sweep.csv\').open(\'w\',newline=\'\',encoding=\'utf-8\') as f:\n        writer=csv.DictWriter(f,list(rows[0]));writer.writeheader();writer.writerows(rows)\n    per_image=[]\n    for p in paths:\n        pred=probs[p.stem]>=threshold\n        gt=strict_label(labels[p.stem],cfg.size)\n        native=np.asarray(Image.open(labels[p.stem]).convert(\'L\'))>127\n        native_pred=cv2.resize(probs[p.stem],(native.shape[1],native.shape[0]),interpolation=cv2.INTER_LINEAR)>=threshold\n        per_image.append(dict(id=p.stem,dice_05=dice_score(probs[p.stem]>=0.5,gt),dice=dice_score(pred,gt),\n                              native_dice=dice_score(native_pred,native),\n                              converter_target_proxy_dice=dice_score(pred,converter_label(labels[p.stem])),\n                              predicted_fraction=float(pred.mean()),target_fraction=float(gt.mean())))\n    with (output/\'validation_per_image.csv\').open(\'w\',newline=\'\',encoding=\'utf-8\') as f:\n        writer=csv.DictWriter(f,list(per_image[0]));writer.writeheader();writer.writerows(per_image)\n    report=dict(best_epoch=epoch,threshold=threshold,validation_ids=[p.stem for p in paths],\n                mean_vessel_dice_05=float(np.mean([r[\'dice_05\'] for r in per_image])),\n                mean_vessel_dice=best[\'mean_vessel_dice\'],\n                mean_native_vessel_dice=float(np.mean([r[\'native_dice\'] for r in per_image])),\n                converter_target_proxy_dice=best[\'converter_target_proxy_dice\'],\n                evaluation=\'single held-out fold (4 images), not full five-fold OOF\',\n                limitation=\'Epoch and threshold chosen on the same small validation set; development estimate only.\',\n                private_ground_truth_encoding_verified=False)\n    save_json(output/\'validation_metrics.json\',report)\n    preview(paths,probs,labels,threshold,output/\'validation_preview.png\')\n    return report\n\n\ndef export_submission(model, test, template_ids, cfg, device, root, output, threshold, epoch):\n    white_dir=output/\'masks_512_white_vessels\'\n    black_dir=output/\'masks_native_black_vessels\'\n    white_dir.mkdir(exist_ok=True);black_dir.mkdir(exist_ok=True)\n    probs={p.stem:predict(model,p,cfg,device) for p in test}\n    paths={p.stem:p for p in test}\n    white_paths=[]\n    encoded=[]\n    for image_id in template_ids:\n        prob=probs[image_id]\n        mask=prob>=threshold\n        white_path=white_dir/f\'{image_id}.png\'\n        Image.fromarray((mask*255).astype(np.uint8)).save(white_path)\n        white_paths.append(str(white_path))\n        width,height=Image.open(paths[image_id]).size\n        native=cv2.resize(prob,(width,height),interpolation=cv2.INTER_LINEAR)>=threshold\n        Image.fromarray(np.where(native,0,255).astype(np.uint8)).save(black_dir/f\'{image_id}.png\')\n        value=rle_encode(mask)\n        if not np.array_equal(rle_decode(value),mask):\n            raise AssertionError(\'RLE roundtrip mismatch\')\n        encoded.append(value)\n    # Execute the supplied conversion function on the actual PNG predictions.\n    official=load_converter(root).turn_to_str(white_paths)\n    if encoded != official:\n        raise AssertionError(\'Vectorized encoding differs from the official converter\')\n    target=output/\'submission.csv\'\n    with target.open(\'w\',newline=\'\',encoding=\'utf-8\') as f:\n        writer=csv.writer(f);writer.writerow([\'Id\',\'Predicted\']);writer.writerows(zip(template_ids,official))\n    with target.open(newline=\'\',encoding=\'utf-8\') as f:\n        rows=list(csv.DictReader(f))\n    if [r[\'Id\'] for r in rows]!=template_ids or len(rows)!=20:\n        raise AssertionError(\'Saved submission IDs/count mismatch\')\n    for row in rows:\n        rle_decode(row[\'Predicted\'])\n    np.savez_compressed(output/\'test_probabilities.npz\',**probs)\n    preview(test,probs,None,threshold,output/\'test_preview.png\')\n    save_json(output/\'submission_provenance.json\',dict(\n        csv_sha256=file_sha(target),model_sha256=file_sha(output/\'fulltrain/last.pt\'),\n        train_images=20,epochs=epoch,threshold=threshold,external_training_data=False,pretrained=False,\n        csv_polarity=\'white vessels, following observed training labels and supplied converter\',\n        image_output_polarity=\'masks_native_black_vessels: vessel=0, background=255\',\n        official_converter_exact_match=True,rle_roundtrip_all_20=True,template_order_preserved=True,\n        submitted_to_competition=False))\n    print(f\'Submission saved: {target}\',flush=True)\n    return target\n\n\ndef run(phase=\'full\', data_dir=None, output_dir=None, config=None):\n    cfg=config or Config()\n    if cfg.size!=512 or cfg.patch<16 or cfg.patch%16 or cfg.patch>cfg.size or cfg.base_channels<8 or cfg.base_channels%8:\n        raise ValueError(\'Require size=512, patch divisible by 16 <=512, base_channels divisible by 8.\')\n    if cfg.epochs<1 or cfg.patches_per_epoch<1 or cfg.batch_size<1 or cfg.threads<1 or not 2<=cfg.folds<=20 or not 0<=cfg.fold<cfg.folds:\n        raise ValueError(\'Invalid training configuration\')\n    torch.set_num_threads(cfg.threads)\n    cv2.setNumThreads(0)\n    root=find_data(data_dir)\n    output=Path(output_dir or \'task4_baseline_results\').resolve()\n    output.mkdir(parents=True,exist_ok=True)\n    source_sha=file_sha(Path(__file__)) if \'__file__\' in globals() else PIPELINE_SHA256\n    config_path=output/\'run_config.json\'\n    previous=json.loads(config_path.read_text(encoding=\'utf-8\')) if config_path.exists() else None\n    if previous and (previous[\'config\']!=asdict(cfg) or previous[\'pipeline_sha256\']!=source_sha):\n        raise ValueError(\'Existing output belongs to another config/source. Use a new directory.\')\n    train,test,labels,assignment,ids,summary=audit(root,output,cfg)\n    identity=dict(config=asdict(cfg),data_fingerprint=summary[\'data_fingerprint\'],\n                  pipeline_sha256=source_sha)\n    if previous and previous!=identity:\n        raise ValueError(\'Existing output belongs to another config/data/source. Use a new directory.\')\n    save_json(config_path,identity)\n    if phase==\'audit\':\n        print(json.dumps(summary,ensure_ascii=False,indent=2),flush=True)\n        return output\n    device=torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n    print(f\'Device: {device}; train=16, validation=4; no pretrained weights\',flush=True)\n    val=[p for p in train if assignment[p.stem]==cfg.fold]\n    development=[p for p in train if assignment[p.stem]!=cfg.fold]\n    # A single writer protects checkpoints from two simultaneous runs.\n    lock=output/\'training.lock\'\n    try:\n        descriptor=os.open(lock,os.O_CREAT|os.O_EXCL|os.O_WRONLY)\n    except FileExistsError:\n        raise RuntimeError(f\'Run lock exists: {lock}. Check its PID before removing a stale lock.\')\n    os.write(descriptor,str(os.getpid()).encode());os.close(descriptor)\n    try:\n        model,epoch=train_stage(development,val,labels,cfg,output/\'holdout\',device,cfg.epochs)\n        metrics=validation_report(model,val,labels,cfg,device,output,epoch)\n        print(json.dumps(metrics,ensure_ascii=False,indent=2),flush=True)\n        if phase==\'full\':\n            del model\n            if device.type==\'cuda\':torch.cuda.empty_cache()\n            model,full_epoch=train_stage(train,[],labels,cfg,output/\'fulltrain\',device,epoch)\n            export_submission(model,test,ids,cfg,device,root,output,metrics[\'threshold\'],full_epoch)\n    finally:\n        lock.unlink(missing_ok=True)\n    return output\n\n\ndef cli():\n    parser=argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--phase\',choices=[\'audit\',\'train\',\'full\'],default=\'full\')\n    parser.add_argument(\'--data-dir\')\n    parser.add_argument(\'--output-dir\',default=\'task4_baseline_results\')\n    for key,value in asdict(Config()).items():\n        if isinstance(value,bool):continue\n        parser.add_argument(\'--\'+key.replace(\'_\',\'-\'),type=type(value),default=value)\n    args=vars(parser.parse_args())\n    phase,data_dir,output=args.pop(\'phase\'),args.pop(\'data_dir\'),args.pop(\'output_dir\')\n    run(phase,data_dir,output,Config(**args))\n\n\nif __name__==\'__main__\':\n    cli()\n', 'task4_kaggle_v2.py': '"""Fixed-epoch five-fold ResNet34 U-Net; native pixels, EMA, paired D4 evaluation.\n\nThe notebook embeds this file AND task4_pipeline.py, so no extra source upload is needed.\n"""\nfrom __future__ import annotations\n\nimport copy\nimport csv\nimport hashlib\nimport json\nimport math\nimport os\nimport platform\nimport random\nimport shutil\nimport socket\nimport time\nimport zipfile\nfrom contextlib import contextmanager\nfrom dataclasses import asdict, dataclass\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\nimport torch\nimport torchvision\nfrom PIL import Image\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom torch.utils.data import DataLoader, Dataset\nfrom torchvision.models import ResNet34_Weights, resnet34\n\nimport task4_pipeline as base\n\n\n@dataclass\nclass ConfigV2:\n    seed: int = 42\n    folds: int = 5\n    fold: int = 0                 # audit helper only; run_cv uses all requested folds\n    epochs: int = 80\n    patch: int = 256\n    patches_per_epoch: int = 256\n    batch_size: int = 8\n    encoder_lr: float = 0.0001\n    decoder_lr: float = 0.0007\n    weight_decay: float = 0.0001\n    warmup_epochs: int = 3\n    ema_decay: float = 0.99\n    monitor_every: int = 10\n    threads: int = 4\n    amp: bool = True\n    pretrained: bool = True\n    tta_gain_required: float = 0.001\n\n\ndef audit_v2(root, output, cfg):\n    """Freeze the original ID folds; JPEG decoder changes must not reshuffle folds."""\n    train,test,labels,_,ids,summary=base.audit(root,output,cfg)\n    frozen=[[\'2\',\'6\',\'9\',\'10\'],[\'3\',\'14\',\'17\',\'20\'],[\'1\',\'4\',\'5\',\'11\'],\n            [\'15\',\'16\',\'18\',\'19\'],[\'7\',\'8\',\'12\',\'13\']]\n    assignment={i:fold for fold,group in enumerate(frozen) for i in group}\n    if set(assignment)!={p.stem for p in train}:\n        raise ValueError(\'This frozen task-4 split expects train IDs 1 through 20.\')\n    manifest=output/\'manifest.csv\'\n    with manifest.open(newline=\'\',encoding=\'utf-8\') as f:\n        reader=csv.DictReader(f);fields=reader.fieldnames;rows=list(reader)\n    groups={}\n    for row in rows:\n        if row[\'split\']==\'train\':\n            row[\'fold\']=assignment[row[\'id\']]\n            groups.setdefault(row[\'pixel_sha256\'],set()).add(row[\'fold\'])\n    if any(len(folds)>1 for folds in groups.values()):\n        raise ValueError(\'A decoded duplicate crosses frozen folds; inspect dataset before training.\')\n    with manifest.open(\'w\',newline=\'\',encoding=\'utf-8\') as f:\n        writer=csv.DictWriter(f,fields);writer.writeheader();writer.writerows(rows)\n    # Raw file hashes are stable across JPEG decoder/library versions, unlike pixel hashes.\n    stable=[{k:row[k] for k in [\'split\',\'id\',\'width\',\'height\',\'image_sha256\',\'label_sha256\',\'fold\']}\n            for row in rows]\n    summary.update(data_fingerprint=hashlib.sha256(json.dumps(stable,sort_keys=True).encode()).hexdigest(),\n                   validation_ids=frozen[0],folds=[4]*5,\n                   split_protocol=\'frozen original seed42 ID folds; training seed may vary\')\n    base.save_json(output/\'audit_summary.json\',summary)\n    return train,test,labels,assignment,ids,summary\n\n\ndef read_rgb(path):\n    return np.asarray(Image.open(path).convert(\'RGB\')).copy()\n\n\ndef read_mask(path):\n    return (np.asarray(Image.open(path).convert(\'L\')) > 127).astype(np.float32)\n\n\ndef resize_probability(prob, shape=(512, 512)):\n    # PIL receives a float image: interpolate probabilities, then threshold once.\n    h, w = shape\n    return np.clip(np.asarray(Image.fromarray(np.asarray(prob, np.float32)).resize(\n        (w, h), Image.Resampling.BILINEAR)), 0, 1).copy()\n\n\nclass NativePatches(Dataset):\n    def __init__(self, paths, labels, cfg):\n        self.images = [read_rgb(p) for p in paths]\n        self.masks = [read_mask(labels[p.stem]) for p in paths]\n        self.positives = [np.argwhere(m > 0) for m in self.masks]\n        self.cfg = cfg\n\n    def __len__(self):\n        return self.cfg.patches_per_epoch\n\n    def __getitem__(self, index):\n        i = random.randrange(len(self.images))\n        image, mask = self.images[i], self.masks[i]\n        h, w = mask.shape\n        patch = self.cfg.patch\n        crop = random.randint(int(patch*0.9), min(int(patch*1.1), h, w))\n        if random.random() < 0.3 and len(self.positives[i]):\n            y, x = self.positives[i][random.randrange(len(self.positives[i]))]\n            top, left = int(np.clip(y-crop//2, 0, h-crop)), int(np.clip(x-crop//2, 0, w-crop))\n        else:\n            top, left = random.randint(0, h-crop), random.randint(0, w-crop)\n        image = cv2.resize(image[top:top+crop,left:left+crop], (patch,patch), interpolation=cv2.INTER_LINEAR)\n        mask = cv2.resize(mask[top:top+crop,left:left+crop], (patch,patch), interpolation=cv2.INTER_NEAREST)\n        image, mask = base.paired_d4(image,mask,random.randrange(4),random.random()<0.5)\n        image = image.astype(np.float32)/255\n        image = np.clip((image-0.5)*random.uniform(0.85,1.15)+0.5+random.uniform(-0.04,0.04),0,1)\n        image = image**random.uniform(0.85,1.15)\n        image = np.clip(image*np.array([random.uniform(0.95,1.05) for _ in range(3)],np.float32),0,1)\n        return torch.from_numpy(image.transpose(2,0,1).copy()), torch.from_numpy(mask[None].copy())\n\n\nclass ResNet34UNet(nn.Module):\n    def __init__(self, initial_state=None):\n        super().__init__()\n        self.encoder = resnet34(weights=None)\n        if initial_state is not None:\n            self.encoder.load_state_dict(initial_state,strict=True)\n        self.encoder.fc = nn.Identity()\n        self.shallow = base.ConvBlock(3,32)\n        self.up4 = base.ConvBlock(512+256,256)\n        self.up3 = base.ConvBlock(256+128,128)\n        self.up2 = base.ConvBlock(128+64,64)\n        self.up1 = base.ConvBlock(64+64,48)\n        self.up0 = base.ConvBlock(48+32,32)\n        self.head = nn.Conv2d(32,1,1)\n        self.aux2 = nn.Conv2d(48,1,1)\n        self.aux4 = nn.Conv2d(64,1,1)\n        self.register_buffer(\'mean\',torch.tensor([0.485,0.456,0.406]).view(1,3,1,1))\n        self.register_buffer(\'std\',torch.tensor([0.229,0.224,0.225]).view(1,3,1,1))\n\n    def train(self, mode=True):\n        super().train(mode)\n        # Small batches: keep ImageNet running statistics; affine parameters still train.\n        for module in self.encoder.modules():\n            if isinstance(module,nn.BatchNorm2d):\n                module.eval()\n        return self\n\n    @staticmethod\n    def join(x, skip, block):\n        x = F.interpolate(x,size=skip.shape[-2:],mode=\'bilinear\',align_corners=False)\n        return block(torch.cat([x,skip],1))\n\n    def forward(self, image):\n        x = (image-self.mean)/self.std\n        s0 = self.shallow(x)\n        s1 = self.encoder.relu(self.encoder.bn1(self.encoder.conv1(x)))\n        s2 = self.encoder.layer1(self.encoder.maxpool(s1))\n        s3 = self.encoder.layer2(s2)\n        s4 = self.encoder.layer3(s3)\n        z = self.encoder.layer4(s4)\n        z = self.join(z,s4,self.up4)\n        z = self.join(z,s3,self.up3)\n        z4 = self.join(z,s2,self.up2)\n        z2 = self.join(z4,s1,self.up1)\n        z = self.join(z2,s0,self.up0)\n        return self.head(z),self.aux2(z2),self.aux4(z4)\n\n\ndef objective(outputs, target):\n    losses = []\n    for logits in outputs:\n        truth = target if logits.shape[-2:]==target.shape[-2:] else F.interpolate(\n            target,size=logits.shape[-2:],mode=\'area\')\n        losses.append(base.segmentation_loss(logits,truth))\n    return (losses[0]+0.1*losses[1]+0.2*losses[2])/1.3\n\n\n@torch.no_grad()\ndef update_ema(ema, model, decay):\n    for target, source in zip(ema.parameters(),model.parameters()):\n        target.lerp_(source,1-decay)\n    for target, source in zip(ema.buffers(),model.buffers()):\n        target.copy_(source)\n\n\ndef transform_d4(x, rotation, flip):\n    x = torch.rot90(x,rotation,(-2,-1))\n    return torch.flip(x,(-1,)) if flip else x\n\n\ndef inverse_d4(x, rotation, flip):\n    if flip:\n        x = torch.flip(x,(-1,))\n    return torch.rot90(x,-rotation,(-2,-1))\n\n\ndef window_starts(length, patch, stride):\n    if length < patch:\n        raise ValueError(\'Pad before calculating window starts\')\n    starts = list(range(0,length-patch+1,stride))\n    if starts[-1]!=length-patch:\n        starts.append(length-patch)\n    return starts\n\n\n@torch.inference_mode()\ndef predict_pair(model, image, cfg, device, with_tta=True):\n    """Patchwise D4; return native probabilities for none and D4 using identical windows."""\n    model.eval()\n    if isinstance(image,(str,Path)):\n        image = read_rgb(image)\n    h,w = image.shape[:2]\n    pad_h,pad_w = max(0,cfg.patch-h),max(0,cfg.patch-w)\n    image = cv2.copyMakeBorder(image,0,pad_h,0,pad_w,cv2.BORDER_REFLECT_101)\n    height,width = image.shape[:2]\n    coords = [(y,x) for y in window_starts(height,cfg.patch,cfg.patch//2)\n              for x in window_starts(width,cfg.patch,cfg.patch//2)]\n    weight = np.maximum(np.outer(np.hanning(cfg.patch),np.hanning(cfg.patch)),0.05).astype(np.float32)\n    accum = np.zeros((2,height,width),np.float32)\n    weights = np.zeros((height,width),np.float32)\n    transforms = [(0,False)] + ([(r,f) for r in range(4) for f in [False,True] if (r,f)!=(0,False)] if with_tta else [])\n    for start in range(0,len(coords),cfg.batch_size):\n        positions = coords[start:start+cfg.batch_size]\n        batch = np.stack([image[y:y+cfg.patch,x:x+cfg.patch].transpose(2,0,1) for y,x in positions])\n        tensor = torch.from_numpy(batch.copy()).to(device).float()/255\n        plain,total = None,None\n        for rotation,flip in transforms:\n            transformed = transform_d4(tensor,rotation,flip)\n            with torch.autocast(device_type=device.type,enabled=cfg.amp and device.type==\'cuda\'):\n                prediction = model(transformed)[0].float().sigmoid()\n            prediction = inverse_d4(prediction,rotation,flip)\n            if plain is None:\n                plain = prediction\n                total = prediction.clone()\n            else:\n                total.add_(prediction)\n        pair = torch.stack([plain,total/len(transforms)])[:,:,0].cpu().numpy()\n        for i,(y,x) in enumerate(positions):\n            accum[:,y:y+cfg.patch,x:x+cfg.patch] += pair[:,i]*weight\n            weights[y:y+cfg.patch,x:x+cfg.patch] += weight\n    result = accum/weights[None]\n    if not np.isfinite(result).all():\n        raise FloatingPointError(\'Nonfinite inference output\')\n    return {\'none\':result[0,:h,:w].copy(),\'d4\':result[1,:h,:w].copy()}\n\n\ndef resolve_weights(cfg, output, weights_path=None):\n    if not cfg.pretrained:\n        return None,dict(source=\'random initialization; debug/ablation only\',sha256=None)\n    enum = ResNet34_Weights.IMAGENET1K_V1\n    target = Path(weights_path) if weights_path else output/\'pretrained_cache\'/Path(enum.url).name\n    if not target.is_file():\n        if weights_path:\n            raise FileNotFoundError(target)\n        target.parent.mkdir(parents=True,exist_ok=True)\n        print(\'Downloading official ImageNet encoder weights:\',enum.url,flush=True)\n        torch.hub.download_url_to_file(enum.url,str(target),hash_prefix=\'b627a593\',progress=True)\n    digest = base.file_sha(target)\n    if not digest.startswith(\'b627a593\'):\n        raise ValueError(\'Expected official torchvision resnet34-b627a593.pth, not task-finetuned weights.\')\n    state = torch.load(target,map_location=\'cpu\',weights_only=True)\n    probe = resnet34(weights=None)\n    probe.load_state_dict(state,strict=True)\n    del probe\n    cached=output/\'pretrained_cache\'/Path(enum.url).name\n    if target.resolve()!=cached.resolve():\n        cached.parent.mkdir(parents=True,exist_ok=True)\n        shutil.copyfile(target,cached)\n    return state,dict(source=enum.url,sha256=digest,supervision=\'generic ImageNet classification, not vessel segmentation\')\n\n\ndef set_lrs(optimizer, step, steps_per_epoch, cfg):\n    total = cfg.epochs*steps_per_epoch\n    warm = min(cfg.warmup_epochs*steps_per_epoch,max(0,total-1))\n    if warm and step<warm:\n        factor = (step+1)/warm\n    else:\n        progress = (step-warm)/max(1,total-warm-1)\n        factor = 0.02+0.98*0.5*(1+math.cos(math.pi*min(1,progress)))\n    for group,initial in zip(optimizer.param_groups,[cfg.encoder_lr,cfg.decoder_lr]):\n        group[\'lr\'] = initial*factor\n\n\ndef train_fold(paths, val_paths, labels, cfg, fold, folder, initial_state, device):\n    folder.mkdir(parents=True,exist_ok=True)\n    base.seed_all(cfg.seed+1009*fold)\n    dataset = NativePatches(paths,labels,cfg)\n    loader = DataLoader(dataset,batch_size=cfg.batch_size,num_workers=0,shuffle=False,\n                        pin_memory=device.type==\'cuda\',drop_last=False)\n    model = ResNet34UNet(initial_state).to(device)\n    ema = copy.deepcopy(model).eval().requires_grad_(False)\n    encoder = list(model.encoder.parameters())\n    decoder = [p for name,p in model.named_parameters() if not name.startswith(\'encoder.\')]\n    optimizer = torch.optim.AdamW([{\'params\':encoder,\'lr\':cfg.encoder_lr},\n                                 {\'params\':decoder,\'lr\':cfg.decoder_lr}],weight_decay=cfg.weight_decay)\n    scaler = torch.amp.GradScaler(\'cuda\',enabled=cfg.amp and device.type==\'cuda\')\n    history,first,step = [],1,0\n    train_ids,val_ids = [p.stem for p in paths],[p.stem for p in val_paths]\n    last = folder/\'last.pt\'\n    if last.exists():\n        ckpt = torch.load(last,map_location=\'cpu\',weights_only=False)\n        if ckpt[\'train_ids\']!=train_ids or ckpt[\'val_ids\']!=val_ids:\n            raise ValueError(\'Checkpoint split mismatch\')\n        model.load_state_dict(ckpt[\'model\']);ema.load_state_dict(ckpt[\'ema\'])\n        optimizer.load_state_dict(ckpt[\'optimizer\']);scaler.load_state_dict(ckpt[\'scaler\'])\n        history,first,step = ckpt[\'history\'],ckpt[\'epoch\']+1,ckpt[\'step\']\n        base.restore_rng(ckpt[\'rng\'])\n        del ckpt\n        print(f\'fold{fold}: resume at {first}/{cfg.epochs}\',flush=True)\n    for epoch in range(first,cfg.epochs+1):\n        begin = time.perf_counter()\n        model.train()\n        losses = []\n        for image,target in loader:\n            set_lrs(optimizer,step,len(loader),cfg)\n            image,target = image.to(device,non_blocking=True),target.to(device,non_blocking=True)\n            optimizer.zero_grad(set_to_none=True)\n            with torch.autocast(device_type=device.type,enabled=cfg.amp and device.type==\'cuda\'):\n                loss = objective(model(image),target)\n            if not torch.isfinite(loss):\n                raise FloatingPointError(\'Nonfinite training loss\')\n            scaler.scale(loss).backward()\n            scaler.unscale_(optimizer)\n            nn.utils.clip_grad_norm_(model.parameters(),5.0)\n            old_scale = scaler.get_scale()\n            scaler.step(optimizer);scaler.update()\n            if scaler.get_scale()>=old_scale:\n                update_ema(ema,model,cfg.ema_decay)\n            losses.append(float(loss.detach()))\n            step += 1\n        score = None\n        if epoch%cfg.monitor_every==0 or epoch==cfg.epochs:\n            score = float(np.mean([base.dice_score(\n                resize_probability(predict_pair(ema,p,cfg,device,False)[\'none\'])>=0.5,\n                base.strict_label(labels[p.stem])) for p in val_paths]))\n        row = dict(epoch=epoch,loss=float(np.mean(losses)),monitor_dice_05=score,\n                   encoder_lr=optimizer.param_groups[0][\'lr\'],decoder_lr=optimizer.param_groups[1][\'lr\'],\n                   seconds=round(time.perf_counter()-begin,3))\n        history.append(row)\n        if shutil.disk_usage(folder).free<1200*1024**2:\n            raise OSError(\'Less than 1.2 GiB free: preserve old checkpoint and free space before resuming.\')\n        base.atomic_checkpoint(last,dict(model=model.state_dict(),ema=ema.state_dict(),\n            optimizer=optimizer.state_dict(),scaler=scaler.state_dict(),history=history,\n            epoch=epoch,step=step,rng=base.rng_state(),train_ids=train_ids,val_ids=val_ids))\n        with (folder/\'history.csv\').open(\'w\',newline=\'\',encoding=\'utf-8\') as f:\n            writer=csv.DictWriter(f,list(row));writer.writeheader();writer.writerows(history)\n        print(f\'fold{fold} {epoch:03d}/{cfg.epochs} loss={row["loss"]:.4f} \'+\n              (f\'monitor_dice@0.5={score:.5f} \' if score is not None else \'\')+\n              f\'seconds={row["seconds"]}\',flush=True)\n        if epoch==1:\n            print(f\'fold{fold}: measured estimate for this fold training/monitoring: \'\n                  f\'{row["seconds"]*cfg.epochs/60:.1f} min (startup may inflate this; D4 export extra)\',flush=True)\n    # Final EMA is fixed in advance. Validation never selects epochs/checkpoints.\n    base.atomic_checkpoint(folder/\'final_ema.pt\',dict(model=ema.state_dict(),epoch=cfg.epochs,\n                          train_ids=train_ids,val_ids=val_ids,selection=\'predeclared final EMA\'))\n    del model,optimizer,scaler,dataset,loader\n    if device.type==\'cuda\':torch.cuda.empty_cache()\n    return ema\n\n\ndef infer_fold(model, val_paths, test, labels, cfg, folder, device):\n    val_probs,test_probs = {},{}\n    for p in val_paths:\n        pair = predict_pair(model,p,cfg,device)\n        for mode,native in pair.items():\n            val_probs[f\'{mode}_{p.stem}\'] = resize_probability(native)\n            val_probs[f\'native_{mode}_{p.stem}\'] = native\n    np.savez_compressed(folder/\'validation_probabilities.npz\',**val_probs)\n    for p in test:\n        pair = predict_pair(model,p,cfg,device)\n        for mode,native in pair.items():\n            test_probs[f\'{mode}_{p.stem}\'] = resize_probability(native)\n    np.savez_compressed(folder/\'test_probabilities.npz\',**test_probs)\n    clean = {p.stem:base.strict_label(labels[p.stem]) for p in val_paths}\n    summary = dict(epoch=cfg.epochs,val_ids=[p.stem for p in val_paths],\n        none_dice_05=float(np.mean([base.dice_score(val_probs[f\'none_{i}\']>=0.5,clean[i]) for i in clean])),\n        d4_dice_05=float(np.mean([base.dice_score(val_probs[f\'d4_{i}\']>=0.5,clean[i]) for i in clean])))\n    files=[\'history.csv\',\'last.pt\',\'final_ema.pt\',\'validation_probabilities.npz\',\'test_probabilities.npz\']\n    summary[\'sha256\']={name:base.file_sha(folder/name) for name in files}\n    base.save_json(folder/\'completed.json\',summary)\n    print(\'Fold completed:\',summary[\'val_ids\'],summary[\'none_dice_05\'],summary[\'d4_dice_05\'],flush=True)\n\n\ndef threshold_sweep(probs, targets):\n    thresholds = np.round(np.arange(0.1,0.901,0.025),3)\n    rows=[dict(threshold=float(t),mean_dice=float(np.mean([\n        base.dice_score(probs[i]>=t,targets[i]) for i in probs]))) for t in thresholds]\n    best=max(rows,key=lambda r:(r[\'mean_dice\'],-abs(r[\'threshold\']-0.5)))\n    return best,rows\n\n\ndef export_csv(probs, ids, threshold, name, root, output):\n    folder=output/\'masks_512_white_vessels\'/name\n    folder.mkdir(parents=True,exist_ok=True)\n    filenames,encoded=[],[]\n    for i in ids:\n        mask=probs[i]>=threshold\n        path=folder/f\'{i}.png\';Image.fromarray((mask*255).astype(np.uint8)).save(path)\n        value=base.rle_encode(mask)\n        if not np.array_equal(base.rle_decode(value),mask):\n            raise AssertionError(\'RLE mismatch\')\n        filenames.append(str(path));encoded.append(value)\n    official=base.load_converter(root).turn_to_str(filenames)\n    if official!=encoded:raise AssertionError(\'Official converter mismatch\')\n    path=output/f\'{name}.csv\'\n    with path.open(\'w\',newline=\'\',encoding=\'utf-8\') as f:\n        writer=csv.writer(f);writer.writerow([\'Id\',\'Predicted\']);writer.writerows(zip(ids,official))\n    return dict(file=path.name,threshold=threshold,sha256=base.file_sha(path),\n                rows=len(ids),official_converter_exact_match=True,rle_roundtrip=True)\n\n\ndef aggregate(root, output, cfg, train, test, labels, assignment, ids):\n    folds=[output/f\'fold{f}\' for f in range(cfg.folds)]\n    if not all((folder/\'completed.json\').exists() for folder in folds):\n        finished=[f for f,folder in enumerate(folds) if (folder/\'completed.json\').exists()]\n        base.save_json(output/\'partial_cv_summary.json\',dict(completed_folds=finished,complete_oof=False))\n        print(\'Partial CV only; no full-OOF score or ensemble submission. Completed:\',finished,flush=True)\n        return None\n    oof={mode:{} for mode in [\'none\',\'d4\']}\n    native={mode:{} for mode in [\'none\',\'d4\']}\n    test_mean={mode:{i:np.zeros((512,512),np.float64) for i in ids} for mode in oof}\n    for fold,folder in enumerate(folds):\n        metadata=json.loads((folder/\'completed.json\').read_text(encoding=\'utf-8\'))\n        for name,digest in metadata[\'sha256\'].items():\n            if base.file_sha(folder/name)!=digest:raise ValueError(f\'Artifact changed: {folder/name}\')\n        expected={p.stem for p in train if assignment[p.stem]==fold}\n        if set(metadata[\'val_ids\'])!=expected:raise ValueError(\'Validation IDs mismatch\')\n        with np.load(folder/\'validation_probabilities.npz\') as values:\n            for mode in oof:\n                for i in expected:\n                    if i in oof[mode]:raise AssertionError(\'Repeated OOF ID\')\n                    oof[mode][i]=values[f\'{mode}_{i}\'].copy()\n                    native[mode][i]=values[f\'native_{mode}_{i}\'].copy()\n        with np.load(folder/\'test_probabilities.npz\') as values:\n            for mode in oof:\n                for i in ids:test_mean[mode][i]+=values[f\'{mode}_{i}\']/cfg.folds\n    train_ids={p.stem for p in train}\n    if any(set(probs)!=train_ids for probs in oof.values()):raise AssertionError(\'Incomplete OOF coverage\')\n    clean={i:base.strict_label(labels[i]) for i in train_ids}\n    proxy={i:base.converter_label(labels[i]) for i in train_ids}\n    rows,mode_summary,exported=[],{},{}\n    for mode in oof:\n        calibrated,sweep=threshold_sweep(oof[mode],clean)\n        proxy_calibrated,proxy_sweep=threshold_sweep(oof[mode],proxy)\n        threshold=calibrated[\'threshold\']\n        cross_thresholds={}\n        for fold in range(cfg.folds):\n            other={j:oof[mode][j] for j in train_ids if assignment[j]!=fold}\n            cross_thresholds[fold]=threshold_sweep(other,{j:clean[j] for j in other})[0][\'threshold\']\n        with (output/f\'threshold_sweep_{mode}.csv\').open(\'w\',newline=\'\',encoding=\'utf-8\') as f:\n            writer=csv.DictWriter(f,[\'threshold\',\'clean_dice\',\'converter_proxy_dice\']);writer.writeheader()\n            writer.writerows(dict(threshold=a[\'threshold\'],clean_dice=a[\'mean_dice\'],converter_proxy_dice=b[\'mean_dice\'])\n                             for a,b in zip(sweep,proxy_sweep))\n        per=[]\n        for i in sorted(train_ids,key=int):\n            # Threshold calibration excluding this outer fold: diagnostic, not nested CV.\n            cross_t=cross_thresholds[assignment[i]]\n            gt_native=read_mask(labels[i])>0\n            row=dict(id=i,fold=assignment[i],mode=mode,\n                dice_05=base.dice_score(oof[mode][i]>=0.5,clean[i]),\n                dice_global_threshold=base.dice_score(oof[mode][i]>=threshold,clean[i]),\n                dice_other_fold_threshold=base.dice_score(oof[mode][i]>=cross_t,clean[i]),\n                other_fold_threshold=cross_t,\n                native_dice_05=base.dice_score(native[mode][i]>=0.5,gt_native),\n                converter_proxy_dice=base.dice_score(oof[mode][i]>=threshold,proxy[i]))\n            rows.append(row);per.append(row)\n        mode_summary[mode]=dict(mean_dice_05=float(np.mean([r[\'dice_05\'] for r in per])),\n            global_threshold=threshold,mean_dice_global_threshold=calibrated[\'mean_dice\'],\n            mean_dice_other_fold_threshold=float(np.mean([r[\'dice_other_fold_threshold\'] for r in per])),\n            mean_native_dice_05=float(np.mean([r[\'native_dice_05\'] for r in per])),\n            mean_converter_proxy_dice=float(np.mean([r[\'converter_proxy_dice\'] for r in per])),\n            converter_proxy_best_threshold=proxy_calibrated[\'threshold\'],\n            converter_proxy_best_dice=proxy_calibrated[\'mean_dice\'])\n        exported[mode]=export_csv(test_mean[mode],ids,threshold,f\'submission_{mode}\',root,output)\n    chosen=\'d4\' if mode_summary[\'d4\'][\'mean_dice_05\']>=mode_summary[\'none\'][\'mean_dice_05\']+cfg.tta_gain_required else \'none\'\n    source=output/exported[chosen][\'file\']\n    shutil.copyfile(source,output/\'submission.csv\')\n    # Explicitly named diagnostic candidate; never silently substitute it for clean vessels.\n    proxy_file=export_csv(test_mean[chosen],ids,mode_summary[chosen][\'converter_proxy_best_threshold\'],\n                         \'submission_converter_proxy\',root,output)\n    threshold=mode_summary[chosen][\'global_threshold\']\n    base.preview(test,{p.stem:test_mean[chosen][p.stem] for p in test},None,threshold,output/\'test_preview.png\')\n    fold0=[p for p in train if assignment[p.stem]==0]\n    base.preview(fold0,oof[chosen],labels,threshold,output/\'oof_fold0_preview.png\')\n    black=output/\'masks_native_black_vessels\';black.mkdir(exist_ok=True)\n    for p in test:\n        width,height=Image.open(p).size\n        prob=resize_probability(test_mean[chosen][p.stem],(height,width))\n        Image.fromarray(np.where(prob>=threshold,0,255).astype(np.uint8)).save(black/f\'{p.stem}.png\')\n    with (output/\'oof_per_image.csv\').open(\'w\',newline=\'\',encoding=\'utf-8\') as f:\n        writer=csv.DictWriter(f,list(rows[0]));writer.writeheader();writer.writerows(rows)\n    np.savez_compressed(output/\'oof_probabilities.npz\',**{f\'{mode}_{i}\':p for mode,probs in oof.items() for i,p in probs.items()})\n    np.savez_compressed(output/\'test_ensemble_probabilities.npz\',**{f\'{mode}_{i}\':p.astype(np.float32) for mode,probs in test_mean.items() for i,p in probs.items()})\n    summary=dict(complete_oof=True,unique_oof_images=len(train_ids),epochs=cfg.epochs,\n        checkpoint_selection=\'fixed final EMA; no outer-validation epoch selection\',\n        modes=mode_summary,selected_mode=chosen,tta_gain_required=cfg.tta_gain_required,\n        test_ensemble=\'equal average of five fold models; each trained on 16 images\',\n        baseline_reference=dict(ids=[\'2\',\'6\',\'9\',\'10\'],baseline_dice_05=0.7839949354578135,\n            v2_same_ids_none_dice_05=float(np.mean([r[\'dice_05\'] for r in rows if r[\'id\'] in {\'2\',\'6\',\'9\',\'10\'} and r[\'mode\']==\'none\'])),\n            note=\'Matched subset only; baseline did not produce full five-fold OOF.\'),\n        selection_caveat=\'TTA mode and global threshold use OOF labels. Tuned scores are development estimates. \'\n                         \'Other-fold calibration is diagnostic, not a nested independent test.\',\n        private_target_preprocessing_verified=False,submitted_to_competition=False)\n    base.save_json(output/\'cv_summary.json\',summary)\n    base.save_json(output/\'submission_provenance.json\',dict(chosen_mode=chosen,\n        primary_csv_sha256=base.file_sha(output/\'submission.csv\'),alternatives=exported,\n        converter_proxy_candidate=proxy_file,fold_models=[base.file_sha(p/\'final_ema.pt\') for p in folds],\n        calibrated_on=\'20 OOF clean masks; proxy candidate separately calibrated on JPEG converter targets\',\n        image_polarity=\'CSV white vessels; separate native-size PNGs have vessel=0, background=255\',\n        official_converter_exact_match=True,external_vessel_training_data=False))\n    print(json.dumps(summary,ensure_ascii=False,indent=2),flush=True)\n    return summary\n\n\ndef make_backups(output, full=True):\n    """Portable analysis archive; optional full archive includes model and resume states."""\n    output=Path(output)\n    files=[p for p in output.rglob(\'*\') if p.is_file() and p.suffix not in {\'.zip\',\'.tmp\'} and p.name!=\'training.lock\']\n    light=[p for p in files if p.suffix in {\'.json\',\'.csv\',\'.png\',\'.py\',\'.md\'} and \'pretrained_cache\' not in p.parts\n           and not (p.suffix==\'.png\' and \'masks_512_white_vessels\' in p.parts)]\n    for name,paths in [(\'analysis_results.zip\',light)]+([(\'full_backup.zip\',files)] if full else []):\n        target=output/name;tmp=output/(name+\'.tmp\')\n        estimated=sum(p.stat().st_size for p in paths)\n        if shutil.disk_usage(output).free<estimated+256*1024**2:\n            raise OSError(f\'Not enough free space to create {name}; completed results remain intact.\')\n        with zipfile.ZipFile(tmp,\'w\',compression=zipfile.ZIP_STORED,allowZip64=True) as z:\n            for path in paths:z.write(path,path.relative_to(output).as_posix())\n        with zipfile.ZipFile(tmp) as z:\n            bad=z.testzip()\n            if bad:raise IOError(f\'Archive CRC failed: {bad}\')\n        os.replace(tmp,target)\n        print(\'Backup saved:\',target,f\'{target.stat().st_size/1024**2:.1f} MiB\',flush=True)\n\n\ndef restore_output(source, output):\n    source,output=Path(source).resolve(),Path(output).resolve()\n    if source==output:return\n    if output.exists() and any(output.iterdir()):\n        # Existing working output is the authoritative continuation; do not overlay backup.\n        if (output/\'run_config.json\').exists():return\n        raise ValueError(\'Restore target is not empty\')\n    output.mkdir(parents=True,exist_ok=True)\n    if source.is_dir():\n        shutil.copytree(source,output,dirs_exist_ok=True,ignore=shutil.ignore_patterns(\'*.zip\',\'*.tmp\',\'training.lock\'))\n    elif source.suffix.lower()==\'.zip\':\n        with zipfile.ZipFile(source) as z:\n            for item in z.infolist():\n                resolved=(output/item.filename).resolve()\n                if not resolved.is_relative_to(output):raise ValueError(\'Unsafe archive path\')\n            z.extractall(output)\n        (output/\'training.lock\').unlink(missing_ok=True)\n    else:raise ValueError(\'RESUME_FROM must be an existing output folder or full_backup.zip\')\n\n\n@contextmanager\ndef run_lock(output):\n    lock=output/\'training.lock\'\n    if lock.exists():\n        info=json.loads(lock.read_text())\n        alive=False\n        if info.get(\'host\')==socket.gethostname():\n            if os.name==\'nt\':\n                import ctypes\n                from ctypes import wintypes\n                kernel=ctypes.WinDLL(\'kernel32\',use_last_error=True)\n                kernel.OpenProcess.argtypes=[wintypes.DWORD,wintypes.BOOL,wintypes.DWORD]\n                kernel.OpenProcess.restype=wintypes.HANDLE\n                kernel.GetExitCodeProcess.argtypes=[wintypes.HANDLE,ctypes.POINTER(wintypes.DWORD)]\n                kernel.CloseHandle.argtypes=[wintypes.HANDLE]\n                handle=kernel.OpenProcess(0x1000,False,int(info[\'pid\']))\n                if handle:\n                    code=wintypes.DWORD()\n                    alive=bool(kernel.GetExitCodeProcess(handle,ctypes.byref(code))) and code.value==259\n                    kernel.CloseHandle(handle)\n            else:\n                try:os.kill(info[\'pid\'],0);alive=True\n                except OSError:pass\n        if alive:raise RuntimeError(f\'Output is in use by PID {info["pid"]}\')\n        lock.unlink()\n    descriptor=os.open(lock,os.O_CREAT|os.O_EXCL|os.O_WRONLY)\n    os.write(descriptor,json.dumps(dict(pid=os.getpid(),host=socket.gethostname())).encode());os.close(descriptor)\n    try:yield\n    finally:lock.unlink(missing_ok=True)\n\n\ndef run_cv(data_dir=None, output_dir=\'task4_v2_results\', cfg=None, folds_to_run=(0,1,2,3,4),\n           weights_path=None, resume_from=None, backups=True, full_backup=True):\n    cfg=cfg or ConfigV2()\n    if cfg.folds!=5 or cfg.fold!=0 or cfg.patch<64 or cfg.patch%32 or cfg.patch>512:\n        raise ValueError(\'Require 5 folds, audit fold=0, patch multiple of 32 in [64,512].\')\n    if cfg.epochs<1 or cfg.batch_size<1 or cfg.patches_per_epoch<1 or cfg.monitor_every<1:\n        raise ValueError(\'Invalid training configuration\')\n    if not 0<cfg.ema_decay<1 or len(set(folds_to_run))!=len(folds_to_run) or not set(folds_to_run)<=set(range(5)):\n        raise ValueError(\'Invalid EMA decay/fold list\')\n    torch.set_num_threads(cfg.threads);cv2.setNumThreads(0)\n    output=Path(output_dir).resolve()\n    if resume_from:restore_output(resume_from,output)\n    output.mkdir(parents=True,exist_ok=True)\n    root=base.find_data(data_dir)\n    identity=dict(config=asdict(cfg),source_sha256=base.file_sha(__file__),\n                  baseline_helpers_sha256=base.file_sha(base.__file__))\n    config_path=output/\'run_config.json\'\n    previous=json.loads(config_path.read_text(encoding=\'utf-8\')) if config_path.exists() else None\n    if previous and any(previous.get(k)!=v for k,v in identity.items()):\n        raise ValueError(\'Changed config/source: use a NEW OUTPUT_DIR to preserve prior results.\')\n    with run_lock(output):\n        train,test,labels,assignment,ids,audit=audit_v2(root,output,cfg)\n        identity[\'data_fingerprint\']=audit[\'data_fingerprint\']\n        if previous and previous[\'data_fingerprint\']!=identity[\'data_fingerprint\']:\n            raise ValueError(\'Dataset changed: use a new output directory\')\n        initial_state,provenance=resolve_weights(cfg,output,weights_path)\n        identity[\'pretrained_sha256\']=provenance[\'sha256\']\n        if previous and previous!=identity:raise ValueError(\'Pretrained weights differ from saved run\')\n        base.save_json(config_path,identity)\n        base.save_json(output/\'pretrained_source.json\',provenance)\n        base.save_json(output/\'environment.json\',dict(python=platform.python_version(),torch=torch.__version__,\n            torchvision=torchvision.__version__,opencv=cv2.__version__,numpy=np.__version__,\n            gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,\n            training_device_count=1,torch_cuda=torch.version.cuda))\n        code_dir=output/\'runtime_source\';code_dir.mkdir(exist_ok=True)\n        for p in [Path(__file__),Path(base.__file__)]:\n            if p.resolve()!=(code_dir/p.name).resolve():shutil.copyfile(p,code_dir/p.name)\n        device=torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n        print(\'Device:\',device,\'fixed epochs:\',cfg.epochs,\'folds requested:\',folds_to_run,flush=True)\n        if device.type!=\'cuda\':print(\'CPU detected; formal five-fold training will be slow. Enable GPU.\',flush=True)\n        for fold in folds_to_run:\n            folder=output/f\'fold{fold}\'\n            if (folder/\'completed.json\').exists():\n                print(f\'fold{fold}: completed, reusing verified artifacts\',flush=True)\n                continue\n            development=[p for p in train if assignment[p.stem]!=fold]\n            validation=[p for p in train if assignment[p.stem]==fold]\n            model=train_fold(development,validation,labels,cfg,fold,folder,initial_state,device)\n            infer_fold(model,validation,test,labels,cfg,folder,device)\n            del model\n            if device.type==\'cuda\':torch.cuda.empty_cache()\n        summary=aggregate(root,output,cfg,train,test,labels,assignment,ids)\n        if backups:make_backups(output,full=full_backup)\n    return output,summary\n', 'task4_controlled_ablation.py': '"""OOF-only boundary/EMA replay and paired label-supervision experiments.\n\nNo test-set predictions, competition submissions, or public-score-based selection.\n"""\nfrom __future__ import annotations\n\nimport copy\nimport csv\nimport hashlib\nimport json\nimport math\nimport os\nimport random\nimport shutil\nimport time\nimport zipfile\nfrom dataclasses import asdict, dataclass\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\nimport torch\nfrom PIL import Image, ImageDraw\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom torch.utils.data import DataLoader, Dataset\n\nimport task4_pipeline as base\nimport task4_kaggle_v2 as v2\n\n\nVARIANTS=(\'L0_clean\',\'L1_proxy\',\'L2_soft\')\nCALIBRATION_GRID=(0.05,0.075,0.1,0.125,0.15,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9)\n\n\n@dataclass\nclass AblationConfig(v2.ConfigV2):\n    soft_boundary: float = 0.5\n\n\ndef digest_json(value):\n    return hashlib.sha256(json.dumps(value,sort_keys=True).encode()).hexdigest()\n\n\ndef write_table(path, rows):\n    if not rows:return\n    with Path(path).open(\'w\',newline=\'\',encoding=\'utf-8\') as f:\n        writer=csv.DictWriter(f,list(rows[0]));writer.writeheader();writer.writerows(rows)\n\n\ndef skeleton(mask):\n    try:\n        from skimage.morphology import skeletonize\n    except ImportError as exc:\n        raise RuntimeError(\'Install scikit-image for the preregistered centerline metrics.\') from exc\n    return skeletonize(np.asarray(mask,bool))\n\n\ndef targets(labels):\n    result={}\n    for i,path in labels.items():\n        c,p=base.strict_label(path),base.converter_label(path)\n        if np.any(c&~p):raise ValueError(\'Expected clean mask to be a subset of converter proxy\')\n        sc,sp=skeleton(c),skeleton(p)\n        radius=cv2.distanceTransform(c.astype(np.uint8),cv2.DIST_L2,5)\n        result[i]=dict(clean=c,proxy=p,clean_skeleton=sc,proxy_skeleton=sp,\n                       thin_clean_centerline=sc&(radius<=1.5))\n    return result\n\n\ndef segmentation_metrics(pred, truth, truth_skeleton, pred_skeleton):\n    pred,truth=np.asarray(pred,bool),np.asarray(truth,bool)\n    tp=int((pred&truth).sum());n_pred=int(pred.sum());n_true=int(truth.sum())\n    precision=tp/n_pred if n_pred else (1.0 if not n_true else 0.0)\n    recall=tp/n_true if n_true else 1.0\n    top_precision=float((pred_skeleton&truth).sum()/pred_skeleton.sum()) if pred_skeleton.any() else (1.0 if not n_true else 0.0)\n    top_recall=float((truth_skeleton&pred).sum()/truth_skeleton.sum()) if truth_skeleton.any() else 1.0\n    cldice=2*top_precision*top_recall/(top_precision+top_recall) if top_precision+top_recall else 0.0\n    return dict(dice=base.dice_score(pred,truth),precision=precision,recall=recall,\n                centerline_precision=top_precision,centerline_recall=top_recall,cldice=cldice,\n                foreground_fraction=n_pred/pred.size)\n\n\ndef evaluate_masks(experiment, masks, truth, assignments):\n    rows=[]\n    for i in sorted(masks,key=int):\n        mask=np.asarray(masks[i],bool);sp=skeleton(mask)\n        thin=truth[i][\'thin_clean_centerline\']\n        thin_recall=float((mask&thin).sum()/thin.sum()) if thin.any() else None\n        for target in [\'clean\',\'proxy\']:\n            values=segmentation_metrics(mask,truth[i][target],truth[i][target+\'_skeleton\'],sp)\n            rows.append(dict(experiment=experiment,id=i,fold=assignments[i],target=target,\n                             **values,thin_clean_centerline_recall=thin_recall))\n    return rows\n\n\ndef summarize_rows(rows):\n    numeric=[\'dice\',\'precision\',\'recall\',\'centerline_precision\',\'centerline_recall\',\'cldice\',\n             \'foreground_fraction\',\'thin_clean_centerline_recall\']\n    summaries,folds=[],[]\n    for experiment,target in sorted({(r[\'experiment\'],r[\'target\']) for r in rows}):\n        group=[r for r in rows if r[\'experiment\']==experiment and r[\'target\']==target]\n        assert len({r[\'id\'] for r in group})==len(group)\n        values={key:float(np.mean([r[key] for r in group if r[key] is not None]))\n                if any(r[key] is not None for r in group) else None for key in numeric}\n        fold_values=[]\n        for fold in sorted({r[\'fold\'] for r in group}):\n            subset=[r for r in group if r[\'fold\']==fold]\n            score=float(np.mean([r[\'dice\'] for r in subset]));fold_values.append(score)\n            folds.append(dict(experiment=experiment,target=target,fold=fold,n_images=len(subset),\n                              **{key:float(np.mean([r[key] for r in subset if r[key] is not None]))\n                                 if any(r[key] is not None for r in subset) else None for key in numeric}))\n        summaries.append(dict(experiment=experiment,target=target,n_images=len(group),\n            complete_20_image_oof=len(group)==20,**values,worst_fold_dice=min(fold_values),\n            fold_dice_std=float(np.std(fold_values))))\n    return summaries,folds\n\n\ndef probability_preview(images, truth, probs, output):\n    ids=sorted(probs,key=int)[:4];tile=220\n    canvas=Image.new(\'RGB\',(tile*4,(tile+24)*len(ids)),\'white\');draw=ImageDraw.Draw(canvas)\n    for row,i in enumerate(ids):\n        y=row*(tile+24)\n        p=probs[i]\n        hard=p>=0.4;low=p>=0.1\n        dil=cv2.dilate(hard.astype(np.uint8),cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(3,3)))>0\n        panels=[Image.open(images[i]).convert(\'RGB\')]+[Image.fromarray((m*255).astype(np.uint8)).convert(\'RGB\') for m in [hard,low,dil]]\n        for col,(panel,label) in enumerate(zip(panels,[f\'OOF {i}\',\'p >= 0.4\',\'p >= 0.1\',\'dilate(p >= 0.4)\'])):\n            draw.text((col*tile+3,y+4),label,fill=\'black\')\n            canvas.paste(panel.resize((tile,tile)),(col*tile,y+24))\n    canvas.save(output)\n\n\ndef materialize_previous(source, output):\n    """Use a mounted folder read-only, or extract only the artifacts used by replay."""\n    if source is None:\n        preferred=Path(\'/kaggle/working/task4_v2_results\')\n        roots=[preferred] if (preferred/\'run_config.json\').exists() else []\n        if not roots and Path(\'/kaggle/input\').exists():\n            for p in Path(\'/kaggle/input\').rglob(\'run_config.json\'):\n                try:cfg=json.loads(p.read_text())\n                except (ValueError,OSError):continue\n                if cfg.get(\'source_sha256\')==base.file_sha(v2.__file__) and all((p.parent/f\'fold{f}/validation_probabilities.npz\').exists() for f in range(5)):\n                    roots.append(p.parent)\n        roots=sorted(set(p.resolve() for p in roots))\n        if len(roots)==1:return roots[0]\n        zips=list(Path(\'/kaggle/input\').rglob(\'full_backup.zip\')) if Path(\'/kaggle/input\').exists() else []\n        if not roots and len(zips)==1:source=zips[0]\n        else:raise ValueError(f\'Set PREVIOUS_RUN to the previous complete output folder or full_backup.zip. Folders found: {roots}\')\n    source=Path(source).resolve()\n    if source.is_dir():return source\n    if not source.is_file() or source.suffix.lower()!=\'.zip\':raise ValueError(\'PREVIOUS_RUN must be a folder or full_backup.zip\')\n    cached=output/\'previous_run_cache\';cached.mkdir(parents=True,exist_ok=True)\n    archive_sha=base.file_sha(source);marker=cached/\'archive_identity.json\'\n    if marker.exists():\n        if json.loads(marker.read_text())[\'archive_sha256\']!=archive_sha:raise ValueError(\'Different previous archive: use a new OUTPUT_DIR\')\n        return cached\n    with zipfile.ZipFile(source) as z:\n        names=z.namelist()\n        if len(names)!=len(set(names)):raise ValueError(\'Duplicate ZIP members\')\n        wanted=[]\n        for name in names:\n            dest=(cached/name).resolve()\n            if not dest.is_relative_to(cached):raise ValueError(\'Unsafe archive path\')\n            parts=Path(name).parts\n            if name in {\'run_config.json\',\'pretrained_source.json\'} or (\n                len(parts)==2 and parts[0] in {f\'fold{f}\' for f in range(5)} and parts[1] in {\n                    \'completed.json\',\'last.pt\',\'validation_probabilities.npz\',\'history.csv\'}) or (\n                len(parts)==2 and parts[0]==\'pretrained_cache\' and parts[1]==\'resnet34-b627a593.pth\'):\n                wanted.append(name)\n        required_members={\'run_config.json\'}|{f\'fold{f}/{name}\' for f in range(5) for name in [\'completed.json\',\'validation_probabilities.npz\']}\n        if not required_members<=set(wanted):raise ValueError(\'The analysis-only ZIP cannot support these ablations; use the full backup with all five OOF probability files.\')\n        required=sum(z.getinfo(n).file_size for n in wanted)\n        if shutil.disk_usage(output).free<required+512*1024**2:raise OSError(\'Not enough disk space to extract the required prior artifacts\')\n        for name in wanted:z.extract(name,cached)\n    base.save_json(marker,dict(archive_sha256=archive_sha,extracted_files=wanted))\n    return cached\n\n\ndef validate_previous(previous, assignment, fingerprint, expected=None, require_last=True, expected_epochs=80):\n    identity=json.loads((previous/\'run_config.json\').read_text())\n    if identity[\'source_sha256\']!=base.file_sha(v2.__file__) or identity[\'baseline_helpers_sha256\']!=base.file_sha(base.__file__):\n        raise ValueError(\'Previous source differs from the pinned v2 implementation\')\n    if identity[\'data_fingerprint\']!=fingerprint:raise ValueError(\'Previous run uses different training data/split\')\n    cfg=v2.ConfigV2(**identity[\'config\'])\n    if cfg.epochs!=expected_epochs:raise ValueError(f\'Expected a completed {expected_epochs}-epoch run, found {cfg.epochs}\')\n    records={}\n    for fold in range(5):\n        folder=previous/f\'fold{fold}\'\n        done=json.loads((folder/\'completed.json\').read_text())\n        if done[\'epoch\']!=cfg.epochs or set(done[\'val_ids\'])!={i for i,f in assignment.items() if f==fold}:\n            raise ValueError(f\'Incomplete/wrong split: fold{fold}\')\n        wanted=[\'validation_probabilities.npz\']+([\'last.pt\'] if require_last else [])\n        for name in wanted:\n            if not (folder/name).is_file():raise ValueError(f\'Missing fold{fold}/{name}; use the complete prior output/full backup, not the analysis-only ZIP.\')\n            actual=base.file_sha(folder/name)\n            if actual!=done[\'sha256\'][name]:raise ValueError(f\'Changed previous artifact: fold{fold}/{name}\')\n            if expected is not None and actual!=expected[str(fold)][\'sha256\'][name]:\n                raise ValueError(f\'Previous artifact does not match the returned analysis archive: fold{fold}/{name}\')\n        records[str(fold)]=done\n    return cfg,records\n\n\ndef replay_fold(previous, fold, paths, cfg, device, cache):\n    checkpoint=previous/f\'fold{fold}/last.pt\'\n    key=dict(checkpoint_sha256=base.file_sha(checkpoint),config=asdict(cfg),\n             gpu=torch.cuda.get_device_name(0) if device.type==\'cuda\' else \'cpu\',torch=torch.__version__)\n    marker=cache/f\'fold{fold}_identity.json\';npz=cache/f\'fold{fold}_replay.npz\'\n    if marker.exists() and npz.exists():\n        old=json.loads(marker.read_text())\n        if old[\'identity\']!=key or old[\'npz_sha256\']!=base.file_sha(npz):raise ValueError(\'Replay cache changed; use a new output directory\')\n        with np.load(npz) as arrays:return {k:arrays[k].copy() for k in arrays.files}\n    # Hashes were checked against the known previous run before loading its own resume file.\n    state=torch.load(checkpoint,map_location=\'cpu\',weights_only=False)\n    expected_ids={p.stem for p in paths}\n    if state[\'epoch\']!=cfg.epochs or set(state[\'val_ids\'])!=expected_ids or set(state[\'train_ids\'])&expected_ids:\n        raise ValueError(\'OOF leakage/epoch check failed\')\n    model=v2.ResNet34UNet().to(device);values={}\n    for name,key_name in [(\'replay_ema\',\'ema\'),(\'raw\',\'model\')]:\n        model.load_state_dict(state[key_name]);model.eval()\n        for p in paths:\n            pair=v2.predict_pair(model,p,cfg,device)\n            for mode,prob in pair.items():values[f\'{name}_{mode}_{p.stem}\']=v2.resize_probability(prob)\n    del model,state\n    if device.type==\'cuda\':torch.cuda.empty_cache()\n    cache.mkdir(parents=True,exist_ok=True)\n    np.savez_compressed(npz,**values)\n    base.save_json(marker,dict(identity=key,npz_sha256=base.file_sha(npz)))\n    return values\n\n\ndef run_replay(previous, output, train, labels, assignment, fingerprint, expected=None, raw_replay=True):\n    cfg,records=validate_previous(previous,assignment,fingerprint,expected,raw_replay)\n    truth=targets(labels);cached={};replayed={\'replay_ema\':{},\'raw\':{}};drift=[]\n    device=torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n    if raw_replay and device.type!=\'cuda\':raise RuntimeError(\'Enable GPU for paired EMA/raw replay, or turn RUN_RAW_REPLAY off\')\n    for fold in range(5):\n        val=[p for p in train if assignment[p.stem]==fold]\n        with np.load(previous/f\'fold{fold}/validation_probabilities.npz\') as arrays:\n            for p in val:\n                prob=arrays[f\'d4_{p.stem}\'].copy()\n                if prob.shape!=(512,512) or not np.isfinite(prob).all() or prob.min()<0 or prob.max()>1:raise ValueError(\'Invalid cached OOF probability\')\n                cached[p.stem]=prob\n        if raw_replay:\n            values=replay_fold(previous,fold,val,cfg,device,output/\'replay_cache\')\n            for p in val:\n                for name in replayed:replayed[name][p.stem]=values[f\'{name}_d4_{p.stem}\']\n                delta=np.abs(replayed[\'replay_ema\'][p.stem]-cached[p.stem])\n                drift.append(dict(id=p.stem,fold=fold,mean_absolute_probability_drift=float(delta.mean()),\n                                  max_absolute_probability_drift=float(delta.max())))\n        print(f\'Replay fold{fold} completed; no training\',flush=True)\n    if set(cached)!=set(assignment):raise AssertionError(\'Incomplete OOF coverage\')\n    experiments={\n        \'E1_ema_t04\':{i:p>=0.4 for i,p in cached.items()},\n        \'E1_ema_t01\':{i:p>=0.1 for i,p in cached.items()},\n        \'E1_ema_t04_dilate1\':{i:cv2.dilate((p>=0.4).astype(np.uint8),cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(3,3)))>0 for i,p in cached.items()}}\n    if raw_replay:\n        for name,probs in replayed.items():\n            for t,label in [(0.1,\'t01\'),(0.4,\'t04\')]:experiments[f\'E2_{name}_{label}\']={i:p>=t for i,p in probs.items()}\n    rows=[row for name,masks in experiments.items() for row in evaluate_masks(name,masks,truth,assignment)]\n    summary,folds=summarize_rows(rows)\n    write_table(output/\'replay_per_image.csv\',rows);write_table(output/\'replay_summary.csv\',summary)\n    write_table(output/\'replay_per_fold.csv\',folds);write_table(output/\'ema_replay_drift.csv\',drift)\n    probability_preview({p.stem:p for p in train},truth,cached,output/\'boundary_ablation_preview.png\')\n    base.save_json(output/\'replay_audit.json\',dict(complete_20_image_oof=True,fold_records=records,\n        raw_and_ema_replayed_in_same_environment=raw_replay,previous_probability_hashes_verified=True,\n        training_started=False,competition_csv_generated=False,\n        interpretation=\'Morphology and centerline changes are paired diagnostics; no winner selected from public scores.\'))\n    return summary\n\n\nclass PairedPatches(Dataset):\n    """All label variants receive exactly the same RGB patches and augmentations."""\n    def __init__(self, paths, labels, cfg, fold):\n        self.ids=[p.stem for p in paths];self.cfg=cfg;self.fold=fold\n        self.images=[cv2.resize(v2.read_rgb(p),(512,512),interpolation=cv2.INTER_AREA) for p in paths]\n        self.clean=[base.strict_label(labels[i]).astype(np.float32) for i in self.ids]\n        self.proxy=[base.converter_label(labels[i]).astype(np.float32) for i in self.ids]\n        self.positive=[np.argwhere(c>0) for c in self.clean]\n        self.plans=[]\n\n    def set_epoch(self, epoch):\n        self.plans=[]\n        for index in range(self.cfg.patches_per_epoch):\n            rng=random.Random((self.cfg.seed+1009*self.fold)*10**10+epoch*10**6+index)\n            i=rng.randrange(len(self.ids));patch=self.cfg.patch\n            crop=rng.randint(int(patch*0.9),min(int(patch*1.1),512))\n            if rng.random()<0.3 and len(self.positive[i]):\n                y,x=self.positive[i][rng.randrange(len(self.positive[i]))]\n                top,left=int(np.clip(y-crop//2,0,512-crop)),int(np.clip(x-crop//2,0,512-crop))\n            else:top,left=rng.randint(0,512-crop),rng.randint(0,512-crop)\n            self.plans.append((i,top,left,crop,rng.randrange(4),rng.random()<0.5,\n                rng.uniform(0.85,1.15),rng.uniform(-0.04,0.04),rng.uniform(0.85,1.15),\n                [rng.uniform(0.95,1.05) for _ in range(3)]))\n        return digest_json(self.plans)\n\n    def __len__(self):return self.cfg.patches_per_epoch\n\n    def __getitem__(self,index):\n        i,top,left,crop,rotation,flip,contrast,brightness,gamma,gains=self.plans[index]\n        patch=self.cfg.patch\n        image=cv2.resize(self.images[i][top:top+crop,left:left+crop],(patch,patch),interpolation=cv2.INTER_LINEAR)\n        pair=np.stack([self.clean[i],self.proxy[i]],axis=-1)\n        pair=cv2.resize(pair[top:top+crop,left:left+crop],(patch,patch),interpolation=cv2.INTER_NEAREST)\n        image,pair=base.paired_d4(image,pair,rotation,flip)\n        image=np.clip((image.astype(np.float32)/255-0.5)*contrast+0.5+brightness,0,1)**gamma\n        image=np.clip(image*np.array(gains,np.float32),0,1)\n        return torch.from_numpy(image.transpose(2,0,1).copy()),torch.from_numpy(pair.transpose(2,0,1).copy())\n\n\ndef supervised_loss(outputs, pair, variant, alpha=0.5):\n    c,p=pair[:,0:1],pair[:,1:2]\n    if variant==\'L0_clean\':bce_target,dice_target=c,c\n    elif variant==\'L1_proxy\':bce_target,dice_target=p,p\n    elif variant==\'L2_soft\':bce_target,dice_target=c+alpha*(p-c),c\n    else:raise ValueError(variant)\n    losses=[]\n    for logits in outputs:\n        bt=F.interpolate(bce_target,size=logits.shape[-2:],mode=\'area\')\n        dt=F.interpolate(dice_target,size=logits.shape[-2:],mode=\'area\')\n        prob=logits.float().sigmoid()\n        dice=(2*(prob*dt).sum((1,2,3))+1)/(prob.sum((1,2,3))+dt.sum((1,2,3))+1)\n        losses.append(0.5*F.binary_cross_entropy_with_logits(logits.float(),bt.float())+0.5*(1-dice.mean()))\n    return (losses[0]+0.1*losses[1]+0.2*losses[2])/1.3\n\n\ndef train_label_fold(paths, validation, labels, cfg, fold, variant, folder, initial, device):\n    folder.mkdir(parents=True,exist_ok=True);base.seed_all(cfg.seed+1009*fold)\n    dataset=PairedPatches(paths,labels,cfg,fold)\n    loader=DataLoader(dataset,batch_size=cfg.batch_size,shuffle=False,num_workers=0,pin_memory=device.type==\'cuda\')\n    model=v2.ResNet34UNet(initial).to(device);ema=copy.deepcopy(model).eval().requires_grad_(False)\n    initial_model_hash=digest_json({k:hashlib.sha256(v.detach().cpu().numpy().tobytes()).hexdigest() for k,v in model.state_dict().items()})\n    optimizer=torch.optim.AdamW([{\'params\':model.encoder.parameters(),\'lr\':cfg.encoder_lr},\n        {\'params\':[p for n,p in model.named_parameters() if not n.startswith(\'encoder.\')],\'lr\':cfg.decoder_lr}],weight_decay=cfg.weight_decay)\n    scaler=torch.amp.GradScaler(\'cuda\',enabled=cfg.amp and device.type==\'cuda\')\n    history,first,step=[],1,0;last=folder/\'last.pt\'\n    train_ids,val_ids=[p.stem for p in paths],[p.stem for p in validation]\n    if last.exists():\n        saved=torch.load(last,map_location=\'cpu\',weights_only=False)\n        if saved[\'variant\']!=variant or saved[\'train_ids\']!=train_ids or saved[\'val_ids\']!=val_ids or saved[\'initial_model_hash\']!=initial_model_hash:\n            raise ValueError(\'Label experiment checkpoint mismatch\')\n        model.load_state_dict(saved[\'model\']);ema.load_state_dict(saved[\'ema\'])\n        optimizer.load_state_dict(saved[\'optimizer\']);scaler.load_state_dict(saved[\'scaler\'])\n        first,step,history=saved[\'epoch\']+1,saved[\'step\'],saved[\'history\'];base.restore_rng(saved[\'rng\'])\n        del saved\n        print(f\'{variant}/fold{fold}: resume at {first}/{cfg.epochs}\',flush=True)\n    for epoch in range(first,cfg.epochs+1):\n        begin=time.perf_counter();model.train();plan_hash=dataset.set_epoch(epoch);losses=[]\n        for image,pair in loader:\n            v2.set_lrs(optimizer,step,len(loader),cfg)\n            image,pair=image.to(device,non_blocking=True),pair.to(device,non_blocking=True)\n            optimizer.zero_grad(set_to_none=True)\n            with torch.autocast(device_type=device.type,enabled=cfg.amp and device.type==\'cuda\'):\n                loss=supervised_loss(model(image),pair,variant,cfg.soft_boundary)\n            if not torch.isfinite(loss):raise FloatingPointError(\'Nonfinite label experiment loss\')\n            scaler.scale(loss).backward();scaler.unscale_(optimizer);nn.utils.clip_grad_norm_(model.parameters(),5.0)\n            old_scale=scaler.get_scale();scaler.step(optimizer);scaler.update()\n            if scaler.get_scale()>=old_scale:v2.update_ema(ema,model,cfg.ema_decay)\n            losses.append(float(loss.detach()));step+=1\n        monitor_c,monitor_p=None,None\n        if epoch%cfg.monitor_every==0 or epoch==cfg.epochs:\n            scores=[]\n            for path in validation:\n                image=cv2.resize(v2.read_rgb(path),(512,512),interpolation=cv2.INTER_AREA)\n                prob=v2.predict_pair(ema,image,cfg,device,False)[\'none\']\n                scores.append((base.dice_score(prob>=0.5,base.strict_label(labels[path.stem])),\n                               base.dice_score(prob>=0.5,base.converter_label(labels[path.stem]))))\n            monitor_c,monitor_p=map(float,np.mean(scores,axis=0))\n        row=dict(epoch=epoch,loss=float(np.mean(losses)),monitor_clean_05=monitor_c,monitor_proxy_05=monitor_p,\n                 plan_sha256=plan_hash,seconds=round(time.perf_counter()-begin,3))\n        history.append(row)\n        if shutil.disk_usage(folder).free<1200*1024**2:raise OSError(\'Less than 1.2 GiB free; preserve checkpoint and resume after freeing space\')\n        base.atomic_checkpoint(last,dict(model=model.state_dict(),ema=ema.state_dict(),optimizer=optimizer.state_dict(),\n            scaler=scaler.state_dict(),epoch=epoch,step=step,history=history,rng=base.rng_state(),\n            train_ids=train_ids,val_ids=val_ids,variant=variant,initial_model_hash=initial_model_hash))\n        write_table(folder/\'history.csv\',history)\n        print(f\'{variant}/fold{fold} {epoch:03d}/{cfg.epochs} loss={row["loss"]:.4f} seconds={row["seconds"]}\',flush=True)\n    base.atomic_checkpoint(folder/\'final_ema.pt\',dict(model=ema.state_dict(),epoch=cfg.epochs,\n        variant=variant,train_ids=train_ids,val_ids=val_ids,initial_model_hash=initial_model_hash))\n    values={}\n    for path in validation:\n        image=cv2.resize(v2.read_rgb(path),(512,512),interpolation=cv2.INTER_AREA)\n        for mode,prob in v2.predict_pair(ema,image,cfg,device).items():values[f\'{mode}_{path.stem}\']=prob\n    np.savez_compressed(folder/\'validation_probabilities.npz\',**values)\n    files=[\'last.pt\',\'final_ema.pt\',\'history.csv\',\'validation_probabilities.npz\']\n    base.save_json(folder/\'completed.json\',dict(epoch=cfg.epochs,variant=variant,train_ids=train_ids,val_ids=val_ids,\n        initial_model_hash=initial_model_hash,plan_hashes=[r[\'plan_sha256\'] for r in history],\n        sha256={name:base.file_sha(folder/name) for name in files}))\n    del model,ema,optimizer,scaler,dataset,loader\n    if device.type==\'cuda\':torch.cuda.empty_cache()\n\n\ndef aggregate_labels(output, variants, cfg, labels, assignment):\n    complete=[variant for variant in variants if all((output/variant/f\'fold{fold}/completed.json\').exists() for fold in range(5))]\n    truth=targets(labels);all_rows=[];calibrated=[];sweeps=[]\n    for variant in complete:\n        probs={mode:{} for mode in [\'none\',\'d4\']}\n        for fold in range(5):\n            folder=output/variant/f\'fold{fold}\';done=json.loads((folder/\'completed.json\').read_text())\n            if done[\'epoch\']!=cfg.epochs or set(done[\'val_ids\'])!={i for i,f in assignment.items() if f==fold}:raise ValueError(\'Wrong label OOF fold\')\n            for name,digest in done[\'sha256\'].items():\n                if base.file_sha(folder/name)!=digest:raise ValueError(f\'Changed label experiment artifact: {folder/name}\')\n            with np.load(folder/\'validation_probabilities.npz\') as arrays:\n                for mode in probs:\n                    for i in done[\'val_ids\']:probs[mode][i]=arrays[f\'{mode}_{i}\'].copy()\n        if any(set(p)!=set(assignment) for p in probs.values()):raise AssertionError(\'Incomplete label OOF\')\n        for mode,pred in probs.items():\n            for t in [0.1,0.4,0.5]:\n                all_rows+=evaluate_masks(f\'{variant}_{mode}_t{t:g}\',{i:p>=t for i,p in pred.items()},truth,assignment)\n            for target in [\'clean\',\'proxy\']:\n                grid=[]\n                for t in CALIBRATION_GRID:\n                    fold_scores={f:float(np.mean([base.dice_score(pred[i]>=t,truth[i][target]) for i in pred if assignment[i]==f])) for f in range(5)}\n                    row=dict(variant=variant,mode=mode,target=target,threshold=t,mean_dice=float(np.mean(list(fold_scores.values()))),\n                             worst_fold_dice=min(fold_scores.values()),**{f\'fold{f}\':v for f,v in fold_scores.items()})\n                    grid.append(row);sweeps.append(row)\n                best=max(grid,key=lambda r:(r[\'mean_dice\'],-abs(r[\'threshold\']-0.5)))\n                calibrated.append(dict(**best,selection_caveat=\'OOF-tuned development estimate; not nested or independent validation\'))\n    # Pairing is verified even when only some folds/variants are ready.\n    pairing=[]\n    for fold in range(5):\n        records=[]\n        for variant in variants:\n            path=output/variant/f\'fold{fold}/completed.json\'\n            if path.exists():records.append((variant,json.loads(path.read_text())))\n        if len(records)>1:\n            first=records[0][1]\n            for variant,done in records[1:]:\n                if done[\'initial_model_hash\']!=first[\'initial_model_hash\'] or done[\'plan_hashes\']!=first[\'plan_hashes\']:\n                    raise AssertionError(\'Ablation pairing failed: initial weights or sampled RGB patches differ\')\n            pairing.append(dict(fold=fold,variants=[v for v,_ in records],same_initial_weights=True,same_rgb_plan=True))\n    summary,fold_rows=summarize_rows(all_rows)\n    write_table(output/\'label_per_image.csv\',all_rows);write_table(output/\'label_summary.csv\',summary)\n    write_table(output/\'label_per_fold.csv\',fold_rows);write_table(output/\'label_threshold_grid.csv\',sweeps)\n    write_table(output/\'label_calibration_development.csv\',calibrated)\n    base.save_json(output/\'label_audit.json\',dict(complete_variants=complete,requested_variants=list(variants),\n        pairing=pairing,complete_all_requested=len(complete)==len(variants),\n        no_automatic_winner=True,competition_csv_generated=False,public_scores_used=False,\n        calibration_grid=list(CALIBRATION_GRID),geometry=\'RGB/targets both 512; matched across L0/L1/L2\'))\n    return summary\n\n\ndef analysis_archive(output):\n    path=output/\'ablation_analysis_results.zip\';tmp=output/\'ablation_analysis_results.zip.tmp\'\n    files=[p for p in output.rglob(\'*\') if p.is_file() and p.suffix in {\'.json\',\'.csv\',\'.png\',\'.md\',\'.py\'}\n           and \'previous_run_cache\' not in p.parts and p.name!=\'training.lock\']\n    with zipfile.ZipFile(tmp,\'w\',compression=zipfile.ZIP_DEFLATED) as z:\n        for p in files:z.write(p,p.relative_to(output).as_posix())\n    with zipfile.ZipFile(tmp) as z:\n        if z.testzip():raise IOError(\'Analysis ZIP CRC failed\')\n    os.replace(tmp,path)\n    return path\n\n\ndef run_ablation(data_dir=None,output_dir=\'task4_ablation_results\',cfg=None,previous_run=None,\n                 run_inference=True,raw_replay=True,run_training=False,variants=VARIANTS,\n                 folds_to_run=(0,1,2,3,4),weights_path=None,expected_previous=None,resume_from=None):\n    cfg=cfg or AblationConfig()\n    if cfg.patch<64 or cfg.patch%32 or cfg.patch>512 or cfg.soft_boundary!=0.5 or cfg.folds!=5:\n        raise ValueError(\'Require patch multiple of 32 in [64,512], five folds, fixed soft band=0.5\')\n    if cfg.epochs<1 or cfg.batch_size<1 or cfg.patches_per_epoch<1 or cfg.monitor_every<1 or cfg.threads<1:\n        raise ValueError(\'Invalid experiment settings\')\n    if not set(variants)<=set(VARIANTS) or len(set(variants))!=len(variants):raise ValueError(\'Invalid label variants\')\n    if not set(folds_to_run)<=set(range(5)) or len(set(folds_to_run))!=len(folds_to_run):raise ValueError(\'Invalid fold list\')\n    output=Path(output_dir).resolve()\n    if resume_from and not (output/\'ablation_config.json\').exists():\n        source=Path(resume_from)\n        if source.is_dir() and not (source/\'ablation_config.json\').exists():\n            raise ValueError(\'RESUME_ABLATION_FROM must point to this ablation output, not the original v2 run\')\n        if source.is_file():\n            if source.name==\'ablation_analysis_results.zip\':raise ValueError(\'The lightweight analysis ZIP cannot resume training\')\n            with zipfile.ZipFile(source) as z:\n                if \'ablation_config.json\' not in z.namelist():raise ValueError(\'Not an ablation resume archive\')\n        v2.restore_output(source,output)\n    output.mkdir(parents=True,exist_ok=True)\n    torch.set_num_threads(cfg.threads);cv2.setNumThreads(0)\n    root=base.find_data(data_dir)\n    identity=dict(config=asdict(cfg),source_sha256=base.file_sha(__file__),\n        v2_source_sha256=base.file_sha(v2.__file__),base_source_sha256=base.file_sha(base.__file__))\n    path=output/\'ablation_config.json\'\n    previous_identity=json.loads(path.read_text()) if path.exists() else None\n    if previous_identity and any(previous_identity.get(k)!=v for k,v in identity.items()):raise ValueError(\'Changed recipe/source: use a new OUTPUT_DIR\')\n    with v2.run_lock(output):\n        train,_,labels,assignment,_,audit=v2.audit_v2(root,output,cfg)\n        identity[\'data_fingerprint\']=audit[\'data_fingerprint\']\n        if previous_identity and previous_identity!=identity:raise ValueError(\'Dataset changed\')\n        base.save_json(path,identity)\n        sources=output/\'runtime_source\';sources.mkdir(exist_ok=True)\n        for source in [Path(__file__),Path(v2.__file__),Path(base.__file__)]:shutil.copyfile(source,sources/source.name)\n        previous=None;inference_summary=[];label_summary=[]\n        if run_inference:\n            previous=materialize_previous(previous_run,output)\n            inference_summary=run_replay(previous,output,train,labels,assignment,audit[\'data_fingerprint\'],expected_previous,raw_replay)\n        if run_training:\n            device=torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n            if device.type!=\'cuda\':raise RuntimeError(\'Enable GPU for label-supervision training\')\n            if weights_path is None and previous is not None:\n                candidate=previous/\'pretrained_cache/resnet34-b627a593.pth\'\n                if candidate.exists():weights_path=candidate\n            initial,weight_info=v2.resolve_weights(cfg,output,weights_path)\n            base.save_json(output/\'pretrained_source.json\',weight_info)\n            for variant in variants:\n                for fold in folds_to_run:\n                    folder=output/variant/f\'fold{fold}\'\n                    if (folder/\'completed.json\').exists():\n                        print(f\'{variant}/fold{fold}: completed, reusing\',flush=True);continue\n                    development=[p for p in train if assignment[p.stem]!=fold]\n                    validation=[p for p in train if assignment[p.stem]==fold]\n                    train_label_fold(development,validation,labels,cfg,fold,variant,folder,initial,device)\n            label_summary=aggregate_labels(output,variants,cfg,labels,assignment)\n        base.save_json(output/\'experiment_status.json\',dict(inference_done=run_inference,raw_replayed=run_inference and raw_replay,\n            label_training_requested=run_training,requested_folds=list(folds_to_run),requested_variants=list(variants),\n            competition_csv_generated=False,public_scores_used=False,automatic_model_selection=False,\n            inference_summary=inference_summary,label_summary=label_summary))\n        archive=analysis_archive(output)\n        print(\'OOF-only analysis saved:\',archive,flush=True)\n    return output\n', 'task4_recovery.py': '"""Rebuild only native-clean and L1 OOF; fixed fusion diagnostic; explicit backups."""\nfrom __future__ import annotations\nimport hashlib\nimport json\nimport os\nimport shutil\nimport zipfile\nfrom dataclasses import asdict\nfrom pathlib import Path, PurePosixPath\n\nimport numpy as np\nimport torch\n\nimport task4_controlled_ablation as abl\nbase,v2=abl.base,abl.v2\nARMS=(\'native_clean\',\'L1_proxy\')\nRULE=dict(tta=\'d4\',native_center=.1,proxy_center=.4,weight_native=.5,\n          weight_proxy=.5,clip_epsilon=1e-6,fused_threshold=.5,\n          interpretation=\'threshold-centered logits; heuristic alignment, not probability calibration\')\n\n\ndef safe_archive(z):\n    names=z.namelist()\n    if len(names)!=len(set(names)):raise ValueError(\'Duplicate ZIP paths\')\n    for item in z.infolist():\n        p=PurePosixPath(item.filename)\n        if p.is_absolute() or \'..\' in p.parts or \'\\\\\' in item.filename or \':\' in item.filename or ((item.external_attr>>16)&0o170000)==0o120000:\n            raise ValueError(\'Unsafe ZIP path\')\n    if z.testzip() is not None:raise ValueError(\'ZIP CRC failed\')\n\n\ndef restore(source,output):\n    source=Path(source).resolve()\n    if source==output:return\n    if output.is_relative_to(source):raise ValueError(\'Output cannot be inside its resume source\')\n    if output.exists() and any(output.iterdir()):raise ValueError(\'Resume target must be empty\')\n    if source.is_dir():\n        if not (source/\'recovery_config.json\').exists():raise ValueError(\'Not a recovery-stage output\')\n        # OOF bundle is explicitly insufficient to continue training.\n        if not any(source.glob(\'*/fold*/last.pt\')):raise ValueError(\'Resume needs full checkpoints, not the OOF-only bundle\')\n        shutil.copytree(source,output,ignore=shutil.ignore_patterns(\'*.zip\',\'*.tmp\',\'training.lock\',\'backup_receipt.json\'))\n    elif source.is_file() and source.suffix.lower()==\'.zip\':\n        with zipfile.ZipFile(source) as z:\n            safe_archive(z)\n            if \'recovery_config.json\' not in z.namelist() or not any(n.endswith(\'/last.pt\') for n in z.namelist()):\n                raise ValueError(\'Use task4_recovery_full.zip, not an analysis or OOF-only ZIP\')\n            output.mkdir(parents=True,exist_ok=True)\n            for item in z.infolist():\n                if item.filename.endswith((\'.zip\',\'.tmp\',\'training.lock\',\'backup_receipt.json\')):continue\n                z.extract(item,output)\n    else:raise ValueError(\'RESUME_FROM must be a full recovery folder/ZIP\')\n\n\ndef verify_fold(folder,arm,fold,cfg,assignment):\n    done=json.loads((folder/\'completed.json\').read_text())\n    if done[\'epoch\']!=cfg.epochs or set(done[\'val_ids\'])!={i for i,f in assignment.items() if f==fold}:\n        raise ValueError(f\'Wrong epoch/split in {folder}\')\n    if set(done[\'train_ids\'])!=set(assignment)-set(done[\'val_ids\']):raise ValueError(\'Training IDs cross the validation split\')\n    if arm==\'native_clean\' and (done[\'recovery_arm\']!=arm or set(done[\'train_ids\'])!=set(assignment)-set(done[\'val_ids\'])):\n        raise ValueError(\'Native recovery identity mismatch\')\n    if arm==\'L1_proxy\' and done[\'variant\']!=\'L1_proxy\':raise ValueError(\'Expected L1 proxy variant\')\n    for name,digest in done[\'sha256\'].items():\n        if base.file_sha(folder/name)!=digest:raise ValueError(f\'Changed completed artifact: {folder/name}\')\n    with np.load(folder/\'validation_probabilities.npz\',allow_pickle=False) as arrays:\n        expected={f\'{mode}_{i}\' for mode in [\'none\',\'d4\'] for i in done[\'val_ids\']}\n        if set(arrays.files)!=expected:raise ValueError(\'Unexpected OOF keys\')\n        for key in arrays.files:\n            a=arrays[key]\n            if a.shape!=(512,512) or not np.isfinite(a).all() or a.min()<0 or a.max()>1:\n                raise ValueError(\'Invalid OOF probabilities\')\n    return done\n\n\ndef export_native(model,validation,development,cfg,folder,device):\n    values={}\n    for p in validation:\n        for mode,prob in v2.predict_pair(model,p,cfg,device).items():\n            values[f\'{mode}_{p.stem}\']=v2.resize_probability(prob)\n    np.savez_compressed(folder/\'validation_probabilities.npz\',**values)\n    names=[\'history.csv\',\'last.pt\',\'final_ema.pt\',\'validation_probabilities.npz\']\n    base.save_json(folder/\'completed.json\',dict(recovery_arm=\'native_clean\',epoch=cfg.epochs,\n        train_ids=[p.stem for p in development],val_ids=[p.stem for p in validation],\n        checkpoint_selection=\'predeclared final EMA\',test_predictions_generated=False,\n        sha256={n:base.file_sha(folder/n) for n in names}))\n\n\ndef aligned_fusion(a,b):\n    eps=RULE[\'clip_epsilon\']\n    def logit(p):\n        p=np.clip(np.asarray(p,np.float64),eps,1-eps)\n        return np.log(p)-np.log1p(-p)\n    z=.5*(logit(a)-logit(.1))+.5*(logit(b)-logit(.4))\n    return (1/(1+np.exp(-z))).astype(np.float32)\n\n\ndef analyze(output,cfg,labels,assignment,historical=None):\n    ready={arm:[f for f in range(5) if (output/arm/f\'fold{f}/completed.json\').exists()] for arm in ARMS}\n    if any(folds!=list(range(5)) for folds in ready.values()):\n        return dict(complete_20_image_oof=False,complete_folds=ready,summary=[],no_partial_oof_claim=True)\n    probs={arm:{} for arm in ARMS}\n    for arm in ARMS:\n        for fold in range(5):\n            folder=output/arm/f\'fold{fold}\';done=verify_fold(folder,arm,fold,cfg,assignment)\n            with np.load(folder/\'validation_probabilities.npz\',allow_pickle=False) as arrays:\n                for i in done[\'val_ids\']:probs[arm][i]=arrays[f\'d4_{i}\'].copy()\n    if any(set(values)!=set(assignment) for values in probs.values()):raise ValueError(\'Incomplete OOF\')\n    truth=abl.targets(labels)\n    a,b=probs[\'native_clean\'],probs[\'L1_proxy\']\n    fused={i:aligned_fusion(a[i],b[i]) for i in assignment}\n    methods={\'native_d4_t01\':{i:p>=.1 for i,p in a.items()},\n             \'proxy_d4_t04\':{i:p>=.4 for i,p in b.items()},\n             \'fixed_equal_centered_logit\':{i:p>=.5 for i,p in fused.items()}}\n    rows=[]\n    for method,masks in methods.items():rows+=abl.evaluate_masks(method,masks,truth,assignment)\n    summary,folds=abl.summarize_rows(rows)\n    abl.write_table(output/\'fusion_per_image.csv\',rows)\n    abl.write_table(output/\'fusion_per_fold.csv\',folds)\n    abl.write_table(output/\'fusion_summary.csv\',summary)\n    diagnostics=[];paired=[]\n    lookup={(r[\'experiment\'],r[\'target\'],r[\'id\']):r for r in rows}\n    metrics=[\'dice\',\'precision\',\'recall\',\'cldice\',\'thin_clean_centerline_recall\',\'foreground_fraction\']\n    for i in sorted(assignment,key=int):\n        ma,mb=methods[\'native_d4_t01\'][i],methods[\'proxy_d4_t04\'][i]\n        for target in [\'clean\',\'proxy\']:\n            gt=truth[i][target];ea,eb=ma!=gt,mb!=gt\n            union=int((ea|eb).sum());common=int((ea&eb).sum())\n            diagnostics.append(dict(id=i,fold=assignment[i],target=target,native_wrong=int(ea.sum()),\n                proxy_wrong=int(eb.sum()),both_wrong=common,native_wrong_proxy_correct=int((ea&~eb).sum()),\n                proxy_wrong_native_correct=int((eb&~ea).sum()),error_jaccard=common/union if union else 1.,\n                mask_disagreement_fraction=float((ma!=mb).mean())))\n            for reference in [\'native_d4_t01\',\'proxy_d4_t04\']:\n                for key in metrics:\n                    paired.append(dict(reference=reference,id=i,fold=assignment[i],target=target,metric=key,\n                        delta=float(lookup[\'fixed_equal_centered_logit\',target,i][key])-float(lookup[reference,target,i][key])))\n    abl.write_table(output/\'error_complementarity.csv\',diagnostics)\n    abl.write_table(output/\'fusion_paired_per_image.csv\',paired)\n    paired_summary=[]\n    for reference in [\'native_d4_t01\',\'proxy_d4_t04\']:\n        for target in [\'clean\',\'proxy\']:\n            for key in metrics:\n                r=[r for r in paired if r[\'reference\']==reference and r[\'target\']==target and r[\'metric\']==key]\n                ds=[v[\'delta\'] for v in r];fs=[float(np.mean([v[\'delta\'] for v in r if v[\'fold\']==f])) for f in range(5)]\n                paired_summary.append(dict(reference=reference,target=target,metric=key,mean_delta=float(np.mean(ds)),\n                    positive_images=sum(v>0 for v in ds),positive_folds=sum(v>0 for v in fs),\n                    **{f\'fold{f}_delta\':d for f,d in enumerate(fs)}))\n    abl.write_table(output/\'fusion_paired_summary.csv\',paired_summary)\n    np.savez_compressed(output/\'fusion_oof_d4_probabilities.npz\',**fused)\n    if historical is not None:\n        comparison=[]\n        for s in summary:\n            if s[\'experiment\'] in historical:\n                prior=historical[s[\'experiment\']][s[\'target\']]\n                comparison.append(dict(method=s[\'experiment\'],target=s[\'target\'],historical_dice=prior,\n                    rebuilt_dice=s[\'dice\'],difference=s[\'dice\']-prior,\n                    caveat=\'rebuild under fixed recipe; not a new independent holdout; exact weights not guaranteed\'))\n        abl.write_table(output/\'historical_vs_rebuilt.csv\',comparison)\n    return dict(complete_20_image_oof=True,complete_folds=ready,summary=summary,\n        fixed_rule=RULE,weights_or_threshold_search=False,automatic_model_selection=False,\n        public_scores_used=False,test_predictions_generated=False,competition_csv_generated=False,\n        caveat=\'OOF reused for development; aligned logits are a fixed heuristic, not calibrated probabilities or private-score evidence\')\n\n\ndef bundle(output,full=True):\n    """ZIPs remain temporary until downloaded or saved as version outputs."""\n    files=[p for p in output.rglob(\'*\') if p.is_file() and p.suffix not in {\'.zip\',\'.tmp\'}\n           and p.name not in {\'training.lock\',\'backup_receipt.json\'}]\n    result={}\n    for kind in [\'analysis\',\'oof\']+([\'full\'] if full else []):\n        suffixes={\'.json\',\'.csv\',\'.py\',\'.md\'}|({\'.npz\'} if kind==\'oof\' else set())\n        selected=files if kind==\'full\' else [p for p in files if p.suffix in suffixes]\n        required=sum(p.stat().st_size for p in selected)\n        if shutil.disk_usage(output).free<required+256*1024**2:\n            if kind!=\'full\':raise OSError(f\'Insufficient space for the {kind} backup\')\n            result[kind]=dict(saved=False,reason=\'Insufficient free space; OOF bundle exists, download checkpoints separately\')\n            print(\'Full backup skipped: insufficient space. OOF backup is NOT a resumable training backup.\',flush=True)\n            continue\n        path=output/f\'task4_recovery_{kind}.zip\';tmp=path.with_suffix(\'.zip.tmp\')\n        with zipfile.ZipFile(tmp,\'w\',compression=zipfile.ZIP_STORED) as z:\n            for p in selected:z.write(p,p.relative_to(output).as_posix())\n        with zipfile.ZipFile(tmp) as z:\n            if z.testzip() is not None:raise IOError(\'Backup CRC failed\')\n        os.replace(tmp,path)\n        result[kind]=dict(saved=True,path=str(path),bytes=path.stat().st_size,sha256=base.file_sha(path))\n    base.save_json(output/\'backup_receipt.json\',dict(archives=result,\n        downloaded_to_user_computer=False,warning=\'ZIP creation is not persistent storage: download both archives before ending session\'))\n    print(\'BACKUP READY: download task4_recovery_oof.zip and task4_recovery_full.zip; /kaggle/working is temporary.\',flush=True)\n    return result\n\n\ndef run_recovery(data_dir,output_dir,cfg=None,folds_to_run=(0,1,2,3,4),weights_path=None,\n                 resume_from=None,full_backup=True,historical=None,on_backup=None,engineering=False):\n    cfg=cfg or abl.AblationConfig()\n    if not engineering and asdict(cfg)!=asdict(abl.AblationConfig()):\n        raise ValueError(\'Recovery fixes the original seed42/80-epoch recipe; use engineering=True only for tests\')\n    if len(set(folds_to_run))!=len(folds_to_run) or not set(folds_to_run)<=set(range(5)):\n        raise ValueError(\'Invalid folds\')\n    output=Path(output_dir).resolve()\n    if resume_from and not (output/\'recovery_config.json\').exists():restore(resume_from,output)\n    output.mkdir(parents=True,exist_ok=True)\n    identity=dict(config=asdict(cfg),source_sha256=base.file_sha(__file__),\n        ablation_source_sha256=base.file_sha(abl.__file__),v2_source_sha256=base.file_sha(v2.__file__),\n        base_source_sha256=base.file_sha(base.__file__),arms=ARMS,fixed_fusion_rule=RULE,engineering_only=engineering)\n    # Normalize JSON tuple/list representation for resumed comparisons.\n    identity=json.loads(json.dumps(identity))\n    marker=output/\'recovery_config.json\';previous=json.loads(marker.read_text()) if marker.exists() else None\n    if previous and any(previous.get(k)!=v for k,v in identity.items()):raise ValueError(\'Changed source/config; use a new output directory\')\n    torch.set_num_threads(cfg.threads);abl.cv2.setNumThreads(0)\n    with v2.run_lock(output):\n        root=base.find_data(data_dir)\n        train,_,labels,assignment,_,audit=v2.audit_v2(root,output,cfg)\n        identity[\'data_fingerprint\']=audit[\'data_fingerprint\']\n        if previous and previous!=identity:raise ValueError(\'Dataset changed\')\n        base.save_json(marker,identity)\n        import PIL,skimage,platform\n        base.save_json(output/\'execution_environment.json\',dict(python=platform.python_version(),torch=str(torch.__version__),\n            torchvision=str(v2.torchvision.__version__),numpy=np.__version__,opencv=abl.cv2.__version__,\n            pillow=PIL.__version__,skimage=skimage.__version__,gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None))\n        sources=output/\'runtime_source\';sources.mkdir(exist_ok=True)\n        for path in [__file__,abl.__file__,v2.__file__,base.__file__]:shutil.copyfile(path,sources/Path(path).name)\n        device=torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n        if device.type!=\'cuda\':raise RuntimeError(\'Enable GPU for the two-arm recovery\')\n        initial,provenance=v2.resolve_weights(cfg,output,weights_path)\n        if provenance[\'sha256\']!=\'b627a593bcbe140c234610266fe4f8ae95ea42fc881d091c9b6052e6b1d0590f\':\n            raise ValueError(\'Expected the original official ImageNet initializer\')\n        base.save_json(output/\'pretrained_source.json\',provenance)\n        native_cfg=v2.ConfigV2(**{k:asdict(cfg)[k] for k in asdict(v2.ConfigV2())})\n        for arm in ARMS:\n            for fold in folds_to_run:\n                folder=output/arm/f\'fold{fold}\'\n                if (folder/\'completed.json\').exists():\n                    verify_fold(folder,arm,fold,cfg,assignment)\n                    print(f\'{arm}/fold{fold}: verified complete, reusing\',flush=True)\n                    continue\n                dev=[p for p in train if assignment[p.stem]!=fold]\n                val=[p for p in train if assignment[p.stem]==fold]\n                if arm==\'native_clean\':\n                    model=v2.train_fold(dev,val,labels,native_cfg,fold,folder,initial,device)\n                    export_native(model,val,dev,native_cfg,folder,device)\n                    del model\n                    torch.cuda.empty_cache()\n                else:\n                    abl.train_label_fold(dev,val,labels,cfg,fold,\'L1_proxy\',folder,initial,device)\n                verify_fold(folder,arm,fold,cfg,assignment)\n                status=analyze(output,cfg,labels,assignment,historical)\n                status.update(engineering_only=engineering,exact_historical_model_hashes_guaranteed=False,\n                              historical_submissions_overwritten=False)\n                base.save_json(output/\'recovery_status.json\',status)\n                receipt=bundle(output,full_backup)\n                if on_backup is not None:on_backup(output,receipt)\n        status=analyze(output,cfg,labels,assignment,historical)\n        status.update(engineering_only=engineering,exact_historical_model_hashes_guaranteed=False,\n                      historical_submissions_overwritten=False)\n        base.save_json(output/\'recovery_status.json\',status)\n        receipt=bundle(output,full_backup)\n        if on_backup is not None:on_backup(output,receipt)\n    return output\n'}
EXPECTED_FINGERPRINT='703ca1abcfc30fa5d5a5b7dbb8bdba6751aca8a873b6652d841d45abb47ec9a4'
HISTORICAL={'native_d4_t01': {'clean': 0.7450620145020836, 'proxy': 0.8270693241697573}, 'proxy_d4_t04': {'clean': 0.7288474188681274, 'proxy': 0.8249169298036877}}
for name,source in SOURCES.items():
    (RUNTIME_DIR/name).write_text(source,encoding='utf-8',newline='\n')
def load_module(name):
    spec=importlib.util.spec_from_file_location(name,RUNTIME_DIR/(name+'.py'))
    module=importlib.util.module_from_spec(spec);sys.modules[name]=module
    spec.loader.exec_module(module);return module
BASE=load_module('task4_pipeline')
V2=load_module('task4_kaggle_v2')
ABL=load_module('task4_controlled_ablation')
RECOVER=load_module('task4_recovery')


## 3. 数据和GPU检查


In [ ]:
import torch
CFG=ABL.AblationConfig()
ROOT=BASE.find_data(DATA_DIR)
TRAIN,TEST,LABELS,SPLIT,IDS,AUDIT=V2.audit_v2(ROOT,RUNTIME_DIR/'preflight_audit',CFG)
if AUDIT['data_fingerprint']!=EXPECTED_FINGERPRINT:raise ValueError('Dataset differs from the original experiment.')
print('Data:',ROOT)
print('Output:',OUTPUT_DIR)
print('Rebuild only: native-clean + L1-proxy; seed42; 5 folds x 80 epochs each')
print('GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('Fixed fusion rule:',RECOVER.RULE)
print('Existing submissions are preserved. No L0/L2/clDice rerun, test inference, or competition CSV.')
if RUN_TRAINING and not torch.cuda.is_available():raise RuntimeError('Enable GPU before starting recovery.')


## 4. 补跑两个系列

同会话保持配置再次运行可续接；新会话需挂载本轮完整备份，填写RESUME_FROM，输出指向新的空目录。分析ZIP和OOF ZIP不能续训。只有两个系列全部五折完成后才输出完整20图融合诊断。


In [ ]:
def show_backups(output,receipt):
    from IPython.display import display,FileLink
    print('备份仍在临时会话中；请下载到本机。不要只依赖Draft saved或默认Quick Save。')
    for item in receipt.values():
        if item.get('saved'):display(FileLink(item['path']))
RESULT_DIR=None
if RUN_TRAINING:
    RESULT_DIR=RECOVER.run_recovery(ROOT,OUTPUT_DIR,CFG,tuple(FOLDS_TO_RUN),
        weights_path=WEIGHTS_PATH,resume_from=RESUME_FROM,full_backup=SAVE_FULL_BACKUP,
        historical=HISTORICAL,on_backup=show_backups)
else:
    print('Preflight only; no training started.')


## 5. 查看结果并下载备份


In [ ]:
import json
import pandas as pd
if RESULT_DIR is not None:
    status=json.loads((RESULT_DIR/'recovery_status.json').read_text())
    print('Complete 20-image OOF:',status['complete_20_image_oof'])
    for name in ['fusion_summary.csv','fusion_paired_summary.csv','historical_vs_rebuilt.csv']:
        if (RESULT_DIR/name).exists():print(name);display(pd.read_csv(RESULT_DIR/name))
    print('1. 下载task4_recovery_full.zip到本机：权重、优化器和概率，可续训。')
    print('2. 下载task4_recovery_oof.zip到本机：概率和记录，供后续诊断，不能续训。')
    print('3. task4_recovery_analysis.zip是轻量分析包，可发回；它不能代替前两份备份。')
    for name in ['task4_recovery_full.zip','task4_recovery_oof.zip','task4_recovery_analysis.zip','backup_receipt.json']:
        if (RESULT_DIR/name).exists():display(FileLink(str(RESULT_DIR/name)))
